In [0]:
%pip install unidecode bs4 html2text -qq

In [0]:
import json
import base64
import requests
import re
import urllib3
import ast

from unidecode import unidecode
from bs4 import BeautifulSoup
import re
import unicodedata
import hashlib
import html2text
from bs4 import BeautifulSoup
from dateutil.parser import parse
from datetime import datetime

from urllib3.filepost import writer

from pyspark.sql.types import *
from pyspark.sql import SparkSession
import json
from pathlib import Path
import IPython
from pyspark.sql import DataFrame
from pyspark.sql.functions import explode, explode_outer, first
from typing import Union, Protocol, Callable, Optional, Dict, Any, Literal
from datetime import datetime
import pyspark.sql.functions as F
import pyspark.sql.functions
import sys
import pandas as pd
import os
pd.DataFrame.iteritems=pd.DataFrame.items
from pyspark.sql.types import *
from pyspark.sql.functions import col
import gc
import uuid

from functools import reduce
from cryptography.fernet import Fernet

In [0]:
key = "nxyxDEvwBQYIwGWpd7o4kgjvcdjpLWc4wGnl14adf84="
fernet = Fernet(key)

GFDOQA_TEST_PATH = ""

# abfss://{Container[layer]}@{StorageAccount[layer]}/{relative}
# GFDOQA_TEST_PATH = "abfss://qa-automation@cacdgfdoqaautoadls.dfs.core.windows.net/tmp"

TEST_NOT_INITIATED_MESSAGE = f"Test not inited/ Table property not set/"
input_filename = "input_file_name_by_spark"

def init_writer():
    global w
    w = Writer()
#

#passwordmanager.py

class PasswordManager:
    @staticmethod
    def encrypt(plainext):
        return fernet.encrypt(plainext.encode()).decode("utf-8")

    @staticmethod
    def decrypt(encryptedText):
        decode = lambda txt: fernet.decrypt(txt).decode("utf-8")
        if (isinstance(encryptedText,bytes)):
            return decode(encryptedText)
        return decode(str.encode(encryptedText)) if isinstance(encryptedText,str) else ""

#settings.py
class SettingsProto(Protocol):
    enableUDF: bool
    replacelist:dict
    sprk: SparkSession
    loglevel:str
    writeformat:str
    applicationid:str
    summarylog_base:str
    errorlog_base:str
    maxerror:int


    def createTempViews(self, df, viewname) -> None: ...
    def cleartempview(self, viewname) -> None: ...


class Settings:
    CODE_VER = "1.0.0"

    def __init__(self):
        self.writeformat = "delta"
        self.projectname = ""
        self.projectdesc = ""
        self.maxerror = 10
        self.loglevel = "all"
        self.applicationid = ""
        self.output_root = ""
        self.errorlog_base = ""
        self.summarylog_base = ""
        self.replacelist = {}
        self.isdatabricks = False
        self.iswindows = False
        self.tempviews = []
        self.logfile = ""
        self.enableUDF = True
        self.vencrypted_password = True
        self.runallcols = False
        return

    def __str__(self):
        return f'Settings - Code version: {self.CODE_VER}.'

    def projectinfo(self,pname, pdesc):
        self.projectname = pname
        self.projectdesc = pdesc
        return

    def runAllColumns(self,v):
        global runallcols
        runallcols=v

    def encrypted_password(self,v):
        self.vencrypted_password = v
        return

    def enableudf(self,v):
        self.enableUDF = v
        return

    def setmaxerror(self,v):
        try:
            self.maxerror = int(v)
        except:
            self.maxerror = 10
        return

    def setloglevel(self,v:Optional[str]):
        default = "all"
        options = ["all", "summary", "none"]
        if v is not None and isinstance(v, str):
            v = v.lower()
            self.loglevel = v if v in options else default
        else:
            self.loglevel = default
        return

    def _set_directory(self, path):
        path = str(path)
        cond = path.endswith("/") or path.endswith("\\")
        return path[:-1] if cond else path

    def seterrorlogfolder(self, f):
        self.errorlog_base = self._set_directory(f)
        return

    def setsummarylogfolder(self,f):
        self.summarylog_base = self._set_directory(f)
        return

    def getbasepath(self):
        return str(Path(__file__).parent)[:-23]

    def setwriteformat(self,f):
        self.writeformat = str(f).strip()
        if self.writeformat.lower() == "delta":
            self.sprk.sql("SET spark.databricks.delta.formatCheck.enabled=false")
        return

    def setoutputpath(self, path):
        self.output_root = self._set_directory(path)
        return

    def setsparkobject(self, o):
        self.sprk = o
        #self.db_utils = IPython.get_ipython().user_ns["dbutils"]
        return

    def createTempViews(self,df, viewname):
        try:
            df.createOrReplaceTempView(viewname)
            df.cache()
            self.tempviews.append(viewname)
        except Exception as e:
            Writer.logger("exception in createTempViews")
            Writer.logger(e)
        return

    def cleartempviews(self,excludelist=[]):
        for view in self.tempviews[:]:
            if view not in excludelist:
                self.cleartempview(view)
        return

    def cleartempview(self,viewname):
        try:
            self.sprk.sql(f"drop view if exists {viewname}")
            if viewname in self.tempviews:
                self.tempviews.remove(viewname)
        except Exception as e:
            Writer.logger("exception in cleartempview")
            Writer.logger(e)
        return

    class Table:
        name = ''
        colTest = ''
        colTestDataType = ''
        dataTest = ''
        saveTable = ''
        debug = ''
        cleanPLLevel = '*'
        exColsInColTest = []
        exColsInDataTest = []
        srcPK = ''
        srcTableName = ''
        srcQuery = ''
        trgPK = ''
        trgTableName = ''
        trgQuery = ''
        failWhenMatch = ''


#api.py.....
class Rest:
    def __init__(self, settings:Union[Settings,None] = None):
        headers = {"Content-Type": "application/json; charset=UTF-8", "Accept": "application/json"}
        self.set_parameters("REST", "", "get", {}, {}, {}, {}, headers, 10000, 1, True)
        if settings is None:
            settings = Settings()
            settings.setsparkobject(SparkSession.builder.getOrCreate())
            print('Settings instance initiated.')
        self.settings = settings
        return

    def _set_data(self, d):
        return d if isinstance(d, dict) else ast.literal_eval(d)

    def set_parameters(
            self, source:str, endpoint:str, method:Literal['GET','POST'], data:dict, jsondata:dict,
            variables:dict, res_collection:dict, headers:dict, timeout:int, retry:int, ignore_ssl:bool
        ):
        self.__source = source
        self.__endpointURI = endpoint
        self.__apimethod = method
        self.__jdata = self._set_data(data)
        self.__jjson = self._set_data(jsondata)
        self.__variables = self._set_data(variables)
        self.__responsecollection = self._set_data(res_collection)
        self.__jheader = self._set_data(headers)
        self.__timeout = timeout
        self.__retryLimit = retry
        self.__ignoressl = ignore_ssl
        return

    def addheader(self, hn, hv):
        self.__jheader[hn] = hv

    def addauthtoken(self, token):
        self.__jheader["Authorization"] = f"Bearer {token}"
        return

    def addbasicauth(self, username, password):
        token = base64.b64encode(f"{username}:{PasswordManager.decrypt(password)}".encode("utf-8")).decode("ascii")
        self.__jheader["Authorization"] = f"Basic {token}"
        return

    def __resolveparamaters(self):
        vars = self.__variables
        collection = self.__responsecollection
        set_vardata = lambda d, vd: ast.literal_eval(str(d).replace(f"${{{k}}}", vd))
        if len(vars)>0 and len(collection)>0:
            for k, v in vars.items():
                try:
                    vardata = JSONPath.read(collection[v["tablename"]], v["path"])
                except:
                    vardata = v["default"]
                # self.__jdata = re.sub(f"\\${{{k}}}", vardata, self.__jdata)
                self.__jdata = set_vardata(self.__jdata, vardata)
                self.__jheader = set_vardata(self.__jheader, vardata)
        return

    def getresponse(self):
        if self.__ignoressl:
            urllib3.disable_warnings()

        # resolve paramaters
        self.__resolveparamaters()
        if self.__apimethod.lower() == "get":
            method = lambda endpoint, d, json, h, v, t: requests.get(endpoint, h, v, t)
        else: 
            method = lambda endpoint, d, json, h, v, t: requests.post(endpoint, d, json, h, v, t)
        
        retry = 0
        success = False
        while retry<self.__retryLimit or not success:
            resp = method(
                self.__endpointURI, self.__jdata,self.__jjson,
                self.__jheader, (not self.__ignoressl), self.__timeout
            )
            success = resp.status_code in (200, 201)
            retry += 1
        return resp.json()

    def getDataFromPath(self, path):
        return JSONPath.read(self.__response, path)

    def getdataframe(self, responsedata = None):
        spark_obj = self.settings.sprk
        res_data = json.dumps(self.getresponse() if responsedata is None else responsedata)
        return spark_obj.read.option("multiline", "true").json(spark_obj.sparkContext.parallelize([res_data]))

class JSONPath:
    # common vaiable for the class
    # jsonData = {}
    jsonDataList = [{}]
    jsonDataListCopy = [{}]
    logicalOperator = ""

    # read method================================================================================================
    @staticmethod
    def read(json, path):
        # JSONPath.jsonData = json
        o = m = None
        # regexp for split/ replace
        ARRAY_FILTER_CONDITION = "((?<=&&|\\|\\|)|(?=&&|\\|\\|))"
        ARRAY_FILTER = "\\=\\=(?=([^']*'[^']*')*[^']*$)|\\<\\=(?=([^']*'[^']*')*[^']*$)|\\<(?=([^']*'[^']*')*[^']*$)|\\>(?=([^']*'[^']*')*[^']*$)|\\>\\=(?=([^']*'[^']*')*[^']*$)|\\!\\=(?=([^']*'[^']*')*[^']*$)";
        PATH_ARRAY = "[.](?![^\\{\\[]*[\\]\\}])|[.](?![^\\{\\[]*[\\]\\}])"
        PATH_SEQ_ARRAY = "\\]\\[(?=(?:[^\\']*\\'[^\\']*\\')*[^\\']*$)"

        # variables for method
        path = re.sub(PATH_SEQ_ARRAY, "].[", path[2:])
        pathArray = re.compile(PATH_ARRAY).split(path)
        filtersPath = ""
        jsonListFilterConditions = None
        arrayFilterCtr = 0

        # loop on each path seperated by . ouside []
        # modify path if theer is array of array e.g. data[...][0] will be replaced to data[...].[0]
        for p in pathArray:
            filtersPath = ""
            # if path has [] that measn its is an array
            # seperate path and array filter
            if "[" in p and "]" in p:
                filtersPath = p[(p.index("[") + 1):]
                filtersPath = filtersPath[:-1]
                p = p[:p.index("[")]

            # if path is empty or main data is null or if it does not have the key leave temp o (object) untouched
            if len(p.strip()) > 0:
                if json is not None and json[p] is not None:
                    o = json[p]

            #
            if isinstance(o, dict):
                # if given path is json object then get the path and update the main map
                json = o
            elif isinstance(o, list):
                # if given path is jsonarray
                # get arrat from main json object
                JSONPath.jsonDataList = o

                # if array element is requested
                if len(filtersPath.strip()) > 0:
                    # if all elements of array is needed and an index or filter be used after
                    if json is None and len(p) > 0:
                        if isinstance(JSONPath.jsonDataList[0], dict):
                            p1 = p
                            m = [d[p1] for d in JSONPath.jsonDataList]
                            o = m
                            JSONPath.jsonDataList = o

                    # if array element is requested by index
                    if filtersPath.replace("\\]", "").isdigit():
                        o = JSONPath.jsonDataList[int(filtersPath)]
                        try:
                            json = o
                        except Exception as e:
                            pass
                    else:
                        # if array element is requested by conditions
                        # splt by each condition and=>&& or=>||
                        jsonListFilterConditions = list(
                            filter(None, re.compile(ARRAY_FILTER_CONDITION).split(filtersPath)))
                        arrayFilterCtr = 0
                        # process each condition
                        for condition in jsonListFilterConditions:
                            # remove spaces and the begining?
                            condition = condition.strip()
                            if condition.startswith("?"):
                                condition = condition[1:]

                            # remove unused (
                            while condition.startswith("(") and condition.endswith(")"):
                                condition = condition[1:-1]

                            # remove @.
                            if condition.startswith("@."):
                                condition = condition[2:]
                            # splt based on conditional operator, pass it to filter function
                            if "==" in condition:
                                JSONPath.__jsonListFilter(re.compile(ARRAY_FILTER).split(condition)[0],
                                                          re.compile(ARRAY_FILTER).split(condition)[-1],
                                                          "==", arrayFilterCtr)
                            elif "<=" in condition:
                                JSONPath.__jsonListFilter(re.compile(ARRAY_FILTER).split(condition)[0],
                                                          re.compile(ARRAY_FILTER).split(condition)[-1],
                                                          "<=", arrayFilterCtr)
                            elif "<" in condition:
                                JSONPath.__jsonListFilter(re.compile(ARRAY_FILTER).split(condition)[0],
                                                          re.compile(ARRAY_FILTER).split(condition)[-1],
                                                          "<", arrayFilterCtr)
                            elif ">=" in condition:
                                JSONPath.__jsonListFilter(re.compile(ARRAY_FILTER).split(condition)[0],
                                                          re.compile(ARRAY_FILTER).split(condition)[-1],
                                                          ">=", arrayFilterCtr)
                            elif ">" in condition:
                                JSONPath.__jsonListFilter(re.compile(ARRAY_FILTER).split(condition)[0],
                                                          re.compile(ARRAY_FILTER).split(condition)[-1],
                                                          ">", arrayFilterCtr)
                            elif "!=" in condition:
                                JSONPath.__jsonListFilter(re.compile(ARRAY_FILTER).split(condition)[0],
                                                          re.compile(ARRAY_FILTER).split(condition)[-1],
                                                          "!=", arrayFilterCtr)
                            elif condition == "&&" or condition == "||":
                                JSONPath.logicalOperator = condition

                            arrayFilterCtr = arrayFilterCtr + 1

                        # get the filtered array
                        o = JSONPath.jsonDataListCopy

                        # if list has only 1 element convert datatypes
                        # nullify other objects based on instance type

                        if len(JSONPath.jsonDataListCopy) == 1:
                            if isinstance(JSONPath.jsonDataListCopy[0], dict):
                                json = JSONPath.jsonDataListCopy[0]
                                o = json
                                JSONPath.jsonDataList = None
                            elif isinstance(JSONPath.jsonDataListCopy[0], list):
                                JSONPath.jsonDataList = JSONPath.jsonDataListCopy[0]
                                o = JSONPath.jsonDataList
                                json = None
                            elif isinstance(JSONPath.jsonDataListCopy[0], object):
                                o = JSONPath.jsonDataListCopy[0]
                                JSONPath.jsonDataList = None
                                json = None
                        else:
                            # if list has more then 1 element
                            # copy to master list object and nullify other objects
                            JSONPath.jsonDataList = JSONPath.jsonDataListCopy
                            json = None

                else:
                    # this is when all element from an array is the end
                    if json is None and len(p) > 0:
                        if isinstance(JSONPath.jsonDataList[0], dict):
                            p1 = p
                            m = [d[p1] for d in JSONPath.jsonDataList]
                            o = m
            else:
                # return if its is not map or list as it cannot go futher from here
                return str(o)

        # return the final result map or list or value
        return str(o)

    # read method================================================================================================

    @staticmethod
    def __jsonListFilter(colName, colValue, operator, ac):
        # emp mutable element
        jsonDataListNew = [{}]
        # to check if the value is numeric or non-numeric
        isString = "'" in colValue

        if isString:
            colValue = colValue[1:-1]

        # if this is the first condition
        # copy elements from master list to preocess
        # latter this master will be modifyed
        if ac == 0:
            JSONPath.jsonDataListCopy = JSONPath.jsonDataList.copy()
            JSONPath.logicalOperator = ""

        data = JSONPath.jsonDataList
        data_copy = JSONPath.jsonDataListCopy.copy()
        is_like = "*" in colValue or "%" in colValue
        colValueNorm = colValue if isString else float(colValue)
        numeric_ops = {
            "==": lambda a,b: a==b, "!=": lambda a,b: a!=b,
            "<=": lambda a,b: a<=b, ">=": lambda a,b: a>=b,
            "<": lambda a,b: a<b, ">": lambda a,b: a>b
        }
        # if the logical operator is && or the first one
        # from the copied master list filter based on conditions
        if JSONPath.logicalOperator == "" or JSONPath.logicalOperator == "&&":
            if isString:
                if operator == "==":
                    cond = (lambda e: colValueNorm in e[colName]) if is_like else (lambda e: colValueNorm == e[colName])
                    JSONPath.jsonDataListCopy = [d for d in data_copy if cond(d)]
                elif operator == "!=":
                    cond = (lambda e: colValueNorm not in e[colName]) if is_like else (lambda e: colValueNorm != e[colName])
                    JSONPath.jsonDataListCopy = [d for d in data_copy if cond(d)]
            else:
                if operator in numeric_ops:
                    op = numeric_ops[operator]
                    JSONPath.jsonDataListCopy = [d for d in data_copy if op(float(d[colName]), colValueNorm)]
        else:
            if isString:
                if operator == "==":
                    cond = (lambda e: colValueNorm in e[colName]) if is_like else (lambda e: colValueNorm == e[colName])
                    jsonDataListNew = [d for d in data if cond(d)]
                elif operator == "!=":
                    cond = (lambda e: colValueNorm not in e[colName]) if is_like else (lambda e: colValueNorm != e[colName])
                    jsonDataListNew = [d for d in data if cond(d)]
            else:
                if operator in numeric_ops:
                    op = numeric_ops[operator]
                    jsonDataListNew = [d for d in data if op(float(d[colName]), colValueNorm)]

        # merge results with ans and or
        data_copy = JSONPath.jsonDataListCopy.copy()
        JSONPath.jsonDataListCopy = list(filter(lambda e: e not in data_copy, jsonDataListNew)) + data_copy
        return
#api.py.....

#functions.py

class Atom_UDF:
    replacelist = {}
    """
    non spark udf functions
    """
    _settings:Optional[SettingsProto] = None

    @staticmethod
    def __process_str(try_block:Callable[[str], str], s:Optional[str] = None):
        try:
            s = try_block(s)
        except:
            s = "" if s is None else s
        return s

    @classmethod
    def __basic_cleaning(cls, s:str):
        return cls.cleanSplChrs(cls.cleanDoubleSpace(cls.cleantabledelimiters(s)))

    @classmethod
    def set_settings(cls, settings:SettingsProto) -> None:
        cls._settings = settings
        return

    @staticmethod
    def getNodeValue(node, key, defval):
        try:
            val = node.attrib[key]
        except Exception as e:
            val = defval
        return val

    @staticmethod
    def getjsonValue(jobj, key, defval):
        try:
            val =  jobj[key]
        except Exception as e:
            val = defval
        return val

    @staticmethod
    def getjsonbool(jobj, key, defval):
        try:
            val = jobj[key]
        except:
            return defval
        if isinstance(val, str):
            return val.lower() == "true"
        return (val == 1) if isinstance(val, int) else False

    @staticmethod
    def getNodeValueArr(node, index, key, defval):
        try:
            return node[index].attrib[key]
        except Exception as e:
            return defval

    @staticmethod
    def fieldname(t):
        t = Atom_UDF.cleanSplChrs(t)
        t = Atom_UDF.cleanPunctuations(t)
        t = Atom_UDF.cleanDoubleSpace(t)
        t = t.replace(" ", "_")
        return t

    @staticmethod
    def getCommonValues(a, b):
        fn = lambda s: s.lower()
        common = set(map(fn, a)).intersection(set(map(fn, b)))
        return list(common) if common else []

    @staticmethod
    def getdfschema(df):
        get_dict = lambda f: {"col_name": f.name, "col_datatype": f.dataType.simpleString()}
        try:
            val = {field.name.lower(): get_dict(field) for field in df.schema.fields}
        except:
            val = {}
        return val
    
    @staticmethod
    def iif(e, t, f):
        if isinstance(e, str):
            e = e.lower() == "true"
        elif isinstance(e, int):
            e = e == 1
        return t if e else f

    """
    spark udf functions
    """
    @staticmethod
    def exceldate(n):
        if n >= -693593 and n <= 2958465:
            return str(datetime.fromordinal(datetime(1900, 1, 1).toordinal() + n - 2))[:10]
        return n

    # gettimestamp
    # return current date & time
    # "%b %d, %Y %H:%M:%S.%f"
    # "%Y-%m-%d %H:%M:%S.%f"
    @staticmethod
    def gettimestamp():
        dt = datetime.today()
        return {
            "log": dt.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3],
            "report": dt.strftime("%b %d, %Y %H:%M:%S.%f")[:-3],
        }

    @staticmethod
    def gettimestampf(frmt="%Y-%m-%d %H:%M:%S.%f"):
        frmt = "%Y-%m-%d %H:%M:%S.%f" if (frmt is None or len(str(frmt)) == 0) else frmt
        return datetime.today().strftime(frmt)[:-3]

    @staticmethod
    def getdate():
        return datetime.today().strftime("%B %d, %Y")

    # remove html tahs from string
    @classmethod
    def htmlString(cls, inpString:Optional[str] = None):
        fn = lambda s: cls.normalize(BeautifulSoup(s, features="html.parser").get_text())
        return Atom_UDF.__process_str(fn, inpString) if cls._settings.enableUDF else inpString

    # remove double space to one space
    @classmethod
    def cleanDoubleSpace(cls, inpString:Optional[str] = None):
        fn = lambda s: re.sub("\\s+", " ", s)
        return cls.__process_str(fn, inpString) if cls._settings.enableUDF else inpString

    # hive mack_hash function
    @classmethod
    def mask_hash(cls, inpString:Optional[str] = None):
        fn = lambda s: hashlib.md5(s.encode()).hexdigest()
        return cls.__process_str(fn, inpString) if cls._settings.enableUDF else inpString

    # my mask function
    @staticmethod
    def atom_mask(s):
        return PasswordManager.encrypt(s)

    # check if given string is date/time
    @staticmethod
    def isdate(s):
        try:
            parse(s)
            return True
        except:
            return False

    # remove \r aand \n
    @classmethod
    def cleanNewLine(cls, inpString:Optional[str] = None):
        fn = lambda s: s.replace('\r', '').replace('\n', '')
        return cls.__process_str(fn, inpString) if cls._settings.enableUDF else inpString

    # remove \t
    @classmethod
    def cleanTab(cls, inpString:Optional[str] = None):
        fn = lambda s: s.replace('\t', '')
        return cls.__process_str(fn, inpString) if cls._settings.enableUDF else inpString

    # remove \r and \n and \t
    @classmethod
    def cleantabledelimiters(cls, inpString:Optional[str] = None):
        fn = lambda s: cls.cleanTab(cls.cleanNewLine(s))
        return cls.__process_str(fn, inpString) if cls._settings.enableUDF else inpString

    # remove punctuations
    @classmethod
    def cleanPunctuations(cls, inpString:Optional[str] = None):
        fn = lambda s: re.sub("[^\\w\\s]", " ", s)
        return cls.__process_str(fn, inpString) if cls._settings.enableUDF else inpString

    # remove leading 0
    @classmethod
    def trimzero(cls, inpString:Optional[str] = None):
        fn = lambda s: re.sub("^0+(?!$)", "", s)
        return cls.__process_str(fn, inpString) if cls._settings.enableUDF else inpString

    # normalize text
    @classmethod
    def normalize(cls, inpString:Optional[str] = None):
        fn = lambda s: re.sub("\\s+", " ", s.replace('\\r', ' ').replace('\\n', ' ').replace('\\t', ' '))
        fn = lambda s: fn(cls.cleantabledelimiters(unidecode(cls.htmlString(s))))
        return cls.__process_str(fn, inpString) if cls._settings.enableUDF else inpString

    # remove splchars
    @classmethod
    def cleanSplChrs(cls, inpString:Optional[str] = None):
        fn = lambda s: unicodedata.normalize('NFD', s).encode('ascii', 'ignore').decode()
        return cls.__process_str(fn, inpString) if cls._settings.enableUDF else inpString

    # remove \r and \n
    @classmethod
    def cleanString(cls, inpString:Optional[str] = None):
        def fn(s:Optional[str]):
            s = cls.__basic_cleaning(s)
            return s[1:-1] if s.startswith("\"") and s.endswith("\"") else s
        return cls.__process_str(fn, inpString) if cls._settings.enableUDF else inpString

    # clean all
    @classmethod
    def cleanAll(cls, inpString:Optional[str] = None):
        def fn(s:Optional[str]):
            s = cls.cleanPunctuations(cls.__basic_cleaning(s))
            return s[1:-1] if s.startswith("\"") and s.endswith("\"") else s 
        
        return cls.__process_str(fn, inpString) if cls._settings.enableUDF else inpString

    # replace string
    @classmethod
    def replacechr(cls, inpString:str, f:str, r:str):
        fn = lambda s: s.replace(f, r)
        return cls.__process_str(fn, inpString) if cls._settings.enableUDF else inpString

    # replace \r
    @classmethod
    def replacenewline(cls, inpString:str, r:str):
        fn = lambda s: s.replace('\r\n', r).replace('\r', r).replace('\n', r)
        return cls.__process_str(fn, inpString) if cls._settings.enableUDF else inpString

    # replace \t
    @classmethod
    def replacetab(cls, inpString:str, r:str):
        fn = lambda s: s.replace('\t', r)
        return cls.__process_str(fn, inpString) if cls._settings.enableUDF else inpString

    # replace \t
    @classmethod
    def replaceabledelimiters(cls, inpString:str, r:str):
        fn = lambda s: cls.replacetab(cls.replacenewline(s, r), r)
        return cls.__process_str(fn, inpString) if cls._settings.enableUDF else inpString

    @classmethod
    def replacestring(cls, inpString:str):
        if inpString is None:
            return ""
        if cls._settings.enableUDF:
            for fr in cls._settings.replacelist:
                try:
                    inpString = inpString.replace(fr, cls._settings.replacelist.get(fr))
                except:
                    pass
        return inpString

    # pk function to clean pk as much as possible
    @staticmethod
    def pk(inpString, pkLvl="*"):
        if inpString is None:
            return ""
        
        def try_block(fn, s:str, cond:bool = True):
            try:
                val = fn(s) if cond else s
            except:
                val = s
            return val
        
        fns = [
            (lambda s: s.replace('\r', '').replace('\n', '').replace('\t', ''), lambda s: True),
            (lambda s: unicodedata.normalize('NFD', s).encode('ascii', 'ignore').decode(), lambda s: True),
            (lambda s: re.sub('\\s+', ' ', s), lambda s: True),
            (lambda s: "{:.3f}".format(re.sub("^0+(?!$)", "", s)), lambda s: re.match("^(\\d*\\.)?\\d+$", inpString)),
            (lambda s: s[1:-1], lambda s: (s.startswith("'") or s.startswith("\"")) and (s.endswith("'") or s.endswith("\"")))
        ]

        for lvl, (fn, cond) in enumerate(fns):
            if str(lvl) in pkLvl or '*' in pkLvl or 'all' in pkLvl:
                inpString = try_block(fn, inpString, cond(inpString))
        return inpString

    # flatten array to string
    @staticmethod
    def to_arraystring(arraystr="", delimiter=";"):
        fn = lambda arr: delimiter.join(sorted(arr))
        try:
            res = fn(arraystr) if type(arraystr) is str else fn(map(str, arraystr))
        except:
            res = "" if arraystr is None else arraystr
        return res

    # convert string to array
    @staticmethod
    def to_array(inputstr="", delimiter=";"):
        try:
            res = sorted(str(inputstr).split(delimiter))
        except:
            res = [] if inputstr is None else inputstr
        return res

    # function to register all udf's
    @classmethod
    def registeruserfunctions(cls):
        print("registring udf functions....")
        #cls._settings.sprk.udf.register("add_hello", Atom_UDF.addhello, StringType())
        cls._settings.sprk.udf.register("mask_hash", Atom_UDF.mask_hash, StringType())
        cls._settings.sprk.udf.register("pk", Atom_UDF.pk, StringType())
        cls._settings.sprk.udf.register("cleantabledelimiters", Atom_UDF.cleantabledelimiters, StringType())
        cls._settings.sprk.udf.register("htmlstring", Atom_UDF.htmlString, StringType())
        cls._settings.sprk.udf.register("to_array", Atom_UDF.to_array, ArrayType(StringType()))
        cls._settings.sprk.udf.register("to_arraystring", Atom_UDF.to_arraystring, StringType())
        cls._settings.sprk.udf.register("isdate", Atom_UDF.isdate, BooleanType())
        cls._settings.sprk.udf.register("atom_mask", Atom_UDF.atom_mask, StringType())
        cls._settings.sprk.udf.register("gettimestamp", Atom_UDF.gettimestampf, StringType())
        cls._settings.sprk.udf.register("cleandoublespace", Atom_UDF.cleanDoubleSpace, StringType())
        cls._settings.sprk.udf.register("cleanstring", Atom_UDF.cleanString, StringType())
        cls._settings.sprk.udf.register("cleantab", Atom_UDF.cleanTab, StringType())
        cls._settings.sprk.udf.register("cleannewline", Atom_UDF.cleanNewLine, StringType())
        cls._settings.sprk.udf.register("cleanpunctuations", Atom_UDF.cleanPunctuations, StringType())
        cls._settings.sprk.udf.register("trimzero", Atom_UDF.trimzero, StringType())
        cls._settings.sprk.udf.register("cleansplchrs", Atom_UDF.cleanSplChrs, StringType())
        cls._settings.sprk.udf.register("normalize", Atom_UDF.normalize, StringType())
        cls._settings.sprk.udf.register("cleanall", Atom_UDF.cleanAll, StringType())
        cls._settings.sprk.udf.register("replacechr", Atom_UDF.replacechr, StringType())
        cls._settings.sprk.udf.register("replacenewline", Atom_UDF.replacenewline, StringType())
        cls._settings.sprk.udf.register("replacetab", Atom_UDF.replacetab, StringType())
        cls._settings.sprk.udf.register("replaceabledelimiters", Atom_UDF.replaceabledelimiters, StringType())
        cls._settings.sprk.udf.register("replacestring", Atom_UDF.replacestring, StringType())
        print("registring udf functions.... completed")
        return

#functions.py

#testrunner.py
class TestRunner:
    __resdetailsummary = {}
    __resdataquality = {}
    __dqcols = []
    __dqId = 0
    __ressummary = {}
    __responsecollection = {}
    __report = None
    __username = ""
    __applicationid = ""
    __reportfile = ""
    __curdate = None
    # __dtformat1 = "%Y-%m-%d %H:%M:%S.%f"
    # __dtformat2 = "%b %d, %Y %H:%M:%S.%f"
    __src_trg_file_list = []
    __test_init=False
    __tableprop_status=False
    __excelobj=[]

    __tableprops = {"TABLENAME": "", "COLTEST": True, "COLTEST-DATATYPE": True,
                    "DATATEST": True, "SAVETABLE": True, "DEBUG": False, "CLEANPKLEVEL": "*",
                    "REPARTITION": False, "EXCLUDECOLS-COLTEST": [input_filename],
                    "EXCLUDECOLS-DATATEST": [input_filename], "REVERSECHK": False,
                    "FILE_NAME": "NA"}

    def __init__(self, settings:Optional[Settings] = None):
        # INIT_STATUS = True
        self.__src_trg_file_list=[]
        bannertext = ""
        numbr = str(datetime.today().strftime("_%H.%M.%S.%f"))
        yyyymm = str(datetime.today().strftime("%Y%m"))
        # self.__applicationid = str(settings.sprk.sparkSessionId()) + numbr
        self.__applicationid = "app_" + yyyymm + "_" + str(uuid.uuid4()) + numbr
        
        if settings is None:
            settings = Settings()
            settings.setsparkobject(SparkSession.builder.getOrCreate())
            print('Settings instance initiated.')
        self._settings = self.__set_settings(settings)
        # This ensures Settings instance is injested into Atom_UDF class and its static and class methods
        Atom_UDF.set_settings(self._settings)
        Writer.set_settings(self._settings)
        

        bannertext += "    _  _____ ___  __  __   \n"
        bannertext += "   / \\|_   _/ _ \\|  \\/  |  \n"
        bannertext += "  / _ \\ | || | | | |\\/| |  \n"
        bannertext += " / ___ \\| || |_| | |  | |  \n"
        bannertext += f"/_/   \\_\\_| \\___/|_|  |_|    version {self._settings.CODE_VER}.\n"
        bannertext += f"spark version {self._settings.sprk.version}.\n"
        bannertext += f"python version {sys.version}.\n"
        bannertext += "\n"

        Writer.logger(bannertext, True)
        # runlog = bannertext

        ##self.__report = Reporter() dont need this
        ##self.__report.init() dont need this
        self.__dqId = 0
        self.__dqcols = []

        Atom_UDF.replacelist = self._settings.replacelist
        Atom_UDF.registeruserfunctions()
        Writer.logger(f"runid={self._settings.applicationid}", True)
        self.loginfo()
        return
    
    # Set initial configuration for settings
    def __set_settings(self, settings:Settings) -> Settings:
        settings.applicationid = self.__applicationid
        settings.errorlog_base = f"{settings.output_root}/{settings.applicationid}/error_log"
        settings.summarylog_base = f"{settings.output_root}/{settings.applicationid}/summary_log"

        settings.isdatabricks = ("DATABRICKS_RUNTIME_VERSION" in os.environ)
        settings.iswindows = ("Windows_NT" in os.environ)
        if settings.iswindows:
            self.__username = os.getlogin()
        else:
            self.__username = settings.sprk.sql("select current_user()").first()[0]
        settings.username = self.__username
        return settings

    # get samples from dataframe as string
    def __getDFSamples(self, df, samples=1):
        #print("DF Samples - type(df)", type(df))
        try: 
            samples_str = (df._jdf.showString(samples, False, True))
        except:
            samples_str = df.limit(samples).toPandas().to_string(justify="left")
        return samples_str

    # get schema from dataframe
    def __getDFSchema(self, df):
        try:
            schema_str = (df._jdf.schema().treeString())
        except:
            schema_str = (df.schema.simpleString())
        return schema_str

    # add missing default options
    def __adddefaultoptions(self, useroptions, defoptions):
        defoptions.update(useroptions)
        return defoptions

    # log
    def loginfo(self):
        self.__curdate = Atom_UDF.gettimestamp()
        Writer.clear()
        Writer.add(self._settings.applicationid)
        Writer.add(self.__curdate["log"])
        Writer.add(self._settings.projectname)
        Writer.add(self._settings.projectdesc)
        Writer.add(self.__username)
        Writer.add(self._settings.CODE_VER)
        Writer.add(self.__curdate["log"][:4])
        Writer.qa_testinfo()

    def getsummary(self):
        return self.__ressummary

    def settableproperty(
            self, tablename:str = "", coltest:bool = True, coldatatypetest:bool = True, datatest:bool = True, savetable:bool = True,
            debug:bool = False, cleanpklevel:str = "*", repartition:bool = False, excolsincoltest:list = [],
            excolsindatatest:list = [], reversecheck:bool = False, filename:str = "NA"
        ):
        self.__tableprop_status=True
        tablename = tablename.replace(" ", "")

        self.__src_trg_file_list = []
        if input_filename not in excolsincoltest:
            excolsincoltest.append(input_filename)

        if input_filename not in excolsindatatest:
            excolsindatatest.append(input_filename)

        self.__tableprops = {
            "TABLENAME": tablename, "COLTEST": coltest, "COLTEST-DATATYPE": coldatatypetest,
            "DATATEST": datatest, "SAVETABLE": savetable, "DEBUG": debug, "CLEANPKLEVEL": cleanpklevel,
            "REPARTITION": repartition, "EXCLUDECOLS-COLTEST": excolsincoltest,
            "EXCLUDECOLS-DATATEST": excolsindatatest, "REVERSECHK": reversecheck,
            "FILE_NAME": filename
        }
        return self.__tableprops

    def inittest(self):
        cond = len(self._settings.projectname)>=3 and len(self._settings.projectdesc)>=10
        self.__test_init = cond

        self.__src_trg_file_list = []
        #self.__report.setstarttime(self.__curdate) dont need this
        self.__ressummary = {
            'FILE_NAME': self.__tableprops["FILE_NAME"], 'TABLE_COUNT': 0,
            'START_TIME': self.__curdate, 'END_TIME': '',
            'MD_SRC_COL_COUNT': 0, 'MD_TRG_COL_COUNT': 0,
            'COLTEST_ERRORS': 0,   'KEYTEST_ERRORS': 0,
            'DT_SRC_COL_COUNT': 0, 'DT_TRG_COL_COUNT': 0,
            'SRC_REC_COUNT': 0,    'TRG_REC_COUNT': 0,
            'DATATEST_ERRORS': 0,
            'DATAQUALITY_TESTS': -1,  'DATAQUALITY_TEST_PASSED': -1,
            'DATAQUALITY_ERRORS': -1, 'DATAQUALITY_TEST_FAILED': -1
        }

        self.__resdetailsummary = {
            'TABLE_NAME': self.__tableprops["TABLENAME"],
            'START_TIME': Atom_UDF.gettimestamp(),
            'END_TIME': '',
            'COL_TEST': {
                'TEST_TYPE':'',
                'START_STATUS': False, 'END_STATUS': False,
                'START_TIME': '',    'END_TIME': '',
                'SRC_COL_COUNT': 0,  'TRG_COL_COUNT': 0,
                'COLTEST_ERRORS': 0, 'CUR_COLTEST_ERRORS': 0,
                'EXCEPTION': ''},
            'KEY_TEST': {
                'START_STATUS': False, 'END_STATUS': False,
                'START_TIME': '', 'END_TIME': '',
                'MATCH_FOUND': 0,
                'KEYTEST_ERRORS': 0,
                'EXCEPTION': ''
            },
            'DATA_TEST': {
                'START_STATUS': False, 'END_STATUS': False,
                'START_TIME': '',   'END_TIME': '',
                'SRC_COL_COUNT': 0, 'SRC_REC_COUNT': 0,
                'TRG_COL_COUNT': 0, 'TRG_REC_COUNT': 0,
                'FAILED_COLS': 0,
                'DATATEST_ERRORS': 0,
                'EXCEPTION': ''
            },
            'DATAQUALITY': {}
        }
        return
    # select_columns - list of columns to be viewed when returning df
    # stop_at_column - column name to stop flattening at
    def flatten(self, df, select_columns=None, stop_at_column=None):
        # Cache the DataFrame to avoid redundant computations (won't re-execute previous transformations)
        df.cache()
        flattened_columns = set(df.columns)

        while True:
            # Find columns to flatten
            complex_fields = [(field.name, field.dataType) for field in df.schema.fields
                              if isinstance(field.dataType, (StructType, ArrayType, MapType))]

            if not complex_fields:
                break  # No more nested structures

            for col_name, col_type in complex_fields:
                # Stop flattening if the current column is the column to stop at
                if stop_at_column and col_name == stop_at_column:
                    return df.select(*select_columns) if select_columns else df

                if isinstance(col_type, StructType):
                    # Flatten struct: expand into individual columns
                    expanded = [pyspark.sql.functions.col(f"`{col_name}`.`{field.name}`").alias(f"{col_name}_{field.name.replace('.', '_')}")
                                for field in col_type.fields]
                    df = df.select("*", *expanded).drop(col_name)
                    flattened_columns.update([f"{col_name}_{field.name}" for field in col_type.fields])

                # Explode arrays (e.g. phone)
                elif isinstance(col_type, ArrayType):
                    df = df.withColumn(col_name, explode_outer(col_name))

                elif isinstance(col_type, MapType):
                    # Main changes in this section (avoid collect, which makes it more efficient by more than 10s)
                    # Transform maps into key-value columns
                    df = df.withColumn(col_name, explode(col_name))
                    df = df.select("*", pyspark.sql.functions.col(f"{col_name}.key").alias(f"{col_name}_key"), pyspark.sql.functions.col(f"{col_name}.value").alias(f"{col_name}_value"))
                    df = df.groupBy([c for c in df.columns if c not in {f"{col_name}_key", f"{col_name}_value"}]).pivot(f"{col_name}_key").agg(first(f"{col_name}_value"))
                    df = df.drop(col_name)

                # Update flattened columns
                flattened_columns = set(df.columns)

                # Early exit if all select_columns are now present
                if select_columns and all(col in flattened_columns for col in select_columns):
                    return df.select(*select_columns)

        return df.select(*select_columns) if select_columns else df

    # flatten structs
    def __flatten_df(self, df, prefix=None):
        res = False
        fields = ""
        dfschema = df.schema
        for field in dfschema.fields:
            dtype = field.dataType

            if isinstance(dtype, MapType):
                res = True
                keys_df = df.select(F.explode(F.map_keys(F.col(field.name)))).distinct()
                keys = list(map(lambda row: row[0], keys_df.collect()))
                for key in keys:
                    if prefix is None:
                        aliasname=f"{field.name}_{key.strip()}"
                        aliasname=aliasname.replace(".", "_")
                        aliasname=Atom_UDF.fieldname(aliasname)
                        fields += f"{field.name}['`{key}`'] as `{aliasname}`, "
                    else:
                        aliasname = f"{prefix}_{field.name}{key.strip()}"
                        aliasname = aliasname.replace(".", "_")
                        aliasname=Atom_UDF.fieldname(aliasname)
                        fields += f"{field.name}['`{key}`'] as `{aliasname}`, "

            elif isinstance(dtype, StructType):
                res = True
                for k in dtype.fields:
                    aliasname = f"{field.name}_{k.name}"
                    aliasname = aliasname.replace(".", "_")
                    aliasname = Atom_UDF.fieldname(aliasname)
                    fields += f"{field.name}.`{k.name}` as  `{aliasname}`, "
            else:
                if prefix is None:
                    aliasname = Atom_UDF.fieldname(f"{field.name}")
                    aliasname = aliasname.replace(".", "_")
                    aliasname = Atom_UDF.fieldname(aliasname)
                    fields += f"{field.name} as `{aliasname}`, "
                else:
                    aliasname = Atom_UDF.fieldname(f"{prefix}_{field.name}")
                    aliasname = aliasname.replace(".", "_")
                    aliasname = Atom_UDF.fieldname(aliasname)
                    fields += f"{field.name} as `{aliasname}`, "

        return {"RES": res, "COLS": fields[:-2]}

    # flatten array and structs
    def __explodedataframe(self, explodestruct, explodearray, df, tempview, requiredcols=[]):
        chk = True
        res = {}
        sqltext = ""
        ac = 0
        if requiredcols is None:
            requiredcols=[]

        while True:
            if explodestruct:
                if len(requiredcols)>0:
                    col_list=df.columns
                    if all(field in col_list for field in requiredcols):
                        break

                res = self.__flatten_df(df)
                sqltext = f"select {res['COLS']}  from {tempview}"
                df = self._settings.sprk.sql(sqltext)
                self._settings.createTempViews(df, tempview)
                if not res["RES"] and not chk:
                    break

            if explodearray:
                while True:
                    if len(requiredcols) > 0:
                        col_list = df.columns
                        if all(field in col_list for field in requiredcols):
                            break

                    chk = False
                    sqltext = "select "
                    for s in df.schema.fields:
                        if isinstance(s.dataType, ArrayType) and (not chk):
                            chk = True
                            aliasname = Atom_UDF.fieldname(f"{s.name}")
                            sqltext = f"{sqltext}explode_outer(`{s.name}`) as {aliasname}, "
                        else:
                            aliasname = Atom_UDF.fieldname(f"{s.name}")
                            sqltext = f"{sqltext}`{s.name}` as {aliasname}, "

                    if chk:
                        sqltext = f"{sqltext[:-2]} from {tempview}"
                        df = self._settings.sprk.sql(sqltext)
                        self._settings.createTempViews(df, tempview)
                    else:
                        break

                if not explodestruct and not chk:
                    break
            else:
                chk = False

        return df

    def __check_run(self):
        return   self.__tableprop_status and self.__test_init

    #clear excel object
    def clearexcelobj(self):
        global __excelobj
        __excelobj = []

    #create excel object
    def addexcelobj(self, filename, sheetname=None):
        global __excelobj
        if sheetname is not None:
            __excelobj.append({"filename": filename, "sheetname": sheetname})
        else:
            __excelobj.append({"filename": filename, "sheetname": ""})

    #return excel obj
    def getexcelobj(self):
        global __excelobj
        return __excelobj

    #load excel
    def loadexcel(self, tempview, header=0, engine="openpyxl"):
        global __excelobj
        if self.__check_run():
            try:
                excel_flag=False
                Writer.logger(f"load {len(__excelobj)} excel file(s)...", True)

                file_name=""
                sheet_name=""
                if len(__excelobj)>1:
                    datasets = []
                    for excelobj in __excelobj:
                        file_name=excelobj["filename"]
                        sheet_name = excelobj["sheetname"]

                        if (len(str(sheet_name).strip()))>0:
                            dataset = pd.read_excel(file_name, sheet_name, header, engine)
                        else:
                            dataset = pd.read_excel(file_name, header, engine)

                        dataset[input_filename] = f"{file_name}.{sheet_name}"
                        datasets.append(dataset)
                    dataset = self._settings.sprk.createDataFrame(pd.concat(datasets, axis=0, ignore_index=True))
                    datasets = []
                    excel_flag = True
                elif len(__excelobj)==1:
                    file_name = __excelobj[0]["filename"]
                    sheet_name = __excelobj[0]["sheetname"]

                    if (len(str(sheet_name).strip())) > 0:
                        dataset = pd.read_excel(file_name, sheet_name)
                    else:
                        dataset = pd.read_excel(file_name)
                    dataset[input_filename] = f"{file_name}.{sheet_name}"

                    dataset = self._settings.sprk.createDataFrame(dataset)
                    excel_flag = True
                else:
                    Writer.logger(f"excel object not created, cannot load excel file; use clearexcelobj and addexcelobj functions", True)
                    excel_flag = False

                if (excel_flag):
                    dataset.cache()
                    dfcols = dataset.columns
                    self._settings.createTempViews(dataset, tempview)

                    if tempview not in self.__src_trg_file_list:
                        self.__src_trg_file_list.append(tempview)

                    if self.__tableprops["DEBUG"]:
                        Writer.logger(f"Read {len(__excelobj)} excel file(s)")
                        Writer.logger(f"Temp view created as {tempview}")
                        Writer.logger(f"Records => {dataset.count()}")
                        Writer.logger(self.__getDFSchema(dataset))
                        Writer.logger(self.__getDFSamples(dataset))

                    __excelobj=[]
                    return dataset
                __excelobj = []
                return None

            except Exception as e:
                print(e)
                __excelobj = []
                Writer.logger("exception in loadexcel")
                Writer.logger(e)
                return None
        else:
            __excelobj = []
            print(f"{TEST_NOT_INITIATED_MESSAGE} Project name < 3 chars & desc not set < 10 chars.")
            return None
        
    # load csv file
    def loadcsv(self, filename, tempview, useroptions={}):
        if self.__check_run():
            msg = "[loadcsv] Using User provided CSV options" if useroptions else "[loadcsv] Using default CSV options"
            try:
                Writer.logger("load csv files...", True)
                csvdefaultoptions = {
                    "header": "true", "wholeFile": "true", "multiline": "true",
                    "quote": "\"", "escape": "\\", "delimiter": ","
                }
                useroptions = self.__adddefaultoptions(useroptions, csvdefaultoptions)

                print(msg)
                dataset = self._settings.sprk.read.format("csv").options(**useroptions).load(filename) \
                    .withColumn(input_filename, col("_metadata.file_path"))

                print("[loadcsv] Dataset read completed")
                dataset.cache()
                dfcols = dataset.columns
                lastcolumnname = dfcols[-2]

                if '\r' in lastcolumnname or '\n' in lastcolumnname:
                    newcolumnname = lastcolumnname.replace('\r', '').replace('\n', '')
                    dataset.withColumnRenamed(lastcolumnname, newcolumnname)

                # dataset.createOrReplaceTempView(tempview)
                self._settings.createTempViews(dataset, tempview)
										
                if tempview not in self.__src_trg_file_list:
                    self.__src_trg_file_list.append(tempview)
                # self.__explodedataframe(dataset, tempview)

                if self.__tableprops["DEBUG"]:
                    Writer.logger(f"Read csv file from {filename}")
                    Writer.logger(f"Temp view created as {tempview}")
                    Writer.logger(f"Records => {dataset.count()}")
                    Writer.logger(self.__getDFSchema(dataset))
                    Writer.logger(self.__getDFSamples(dataset))
                print("returning dataset from loadcsv")
                return dataset
            except Exception as e:
                Writer.logger("exception in loadcsv")
                Writer.logger(e)
        else:
            print(f"{TEST_NOT_INITIATED_MESSAGE} Project name < 3 chars & desc not set < 10 chars.")
            return None

    # load orc file
    def loadorc(self, filename, tempview, useroptions=None, explodestruct=False, explodearray=False, flatten=False, select_columns=None, stop_at_column=None):
        if self.__check_run():
            try:
                Writer.logger("load orc files...", True)
                if not useroptions:
                    dataset = spark.read.format("orc").load(filename) \
                        .withColumn(input_filename, col("_metadata.file_path"))
                else:
                    dataset = spark.read.format("orc").options(**useroptions).load(filename) \
                        .withColumn(input_filename, col("_metadata.file_path"))

                # dataset.createOrReplaceTempView(tempview)
                dataset.cache()
                self._settings.createTempViews(dataset, tempview)

                if tempview not in self.__src_trg_file_list:
                    self.__src_trg_file_list.append(tempview)

                if explodestruct or explodearray:
                    dataset = self.__explodedataframe(explodestruct, explodearray, dataset, tempview)

                if flatten:
                    dataset = self.flatten(dataset, select_columns, stop_at_column)

                if self.__tableprops["DEBUG"]:
                    Writer.logger(f"Read orc file from {filename}")
                    Writer.logger(f"Temp view created as {tempview}")
                    Writer.logger(f"Records => {dataset.count()}")
                    # dataset.printSchema()
                    Writer.logger(self.__getDFSchema(dataset))
                    # dataset.show(2, truncate=False, vertical=True)
                    Writer.logger(self.__getDFSamples(dataset))

                return dataset
            except Exception as e:
                Writer.logger("exception in loadorc")
                Writer.logger(e)
        else:
            print(f"{TEST_NOT_INITIATED_MESSAGE} Project name < 3 chars & desc not set < 10 chars.")
            return None

    # load xml file
    def loadxml(self, filename, tempview, useroptions=None, explodestruct=False, explodearray=False, flatten=False, select_columns=None, stop_at_column=None):
        if self.__check_run():
            try:
                Writer.logger("load xml files...", True)
                xmldefaultoptions = {"rootTag": "", "rowTag": "", "wholeFile": "true"}

                if not useroptions:
                    dataset = spark.read.format("databricks.spark.xml").options(**xmldefaultoptions).load(
                        filename) \
                        .withColumn(input_filename, col("_metadata.file_path"))
                else:
                    useroptions = self.__adddefaultoptions(useroptions, xmldefaultoptions)
                    dataset = spark.read.format("databricks.spark.xml").options(**useroptions).load(filename) \
                        .withColumn(input_filename, col("_metadata.file_path"))

                # dataset.createOrReplaceTempView(tempview)
                dataset.cache()
                self._settings.createTempViews(dataset, tempview)

                if tempview not in self.__src_trg_file_list:
                    self.__src_trg_file_list.append(tempview)

                if explodestruct or explodearray:
                    dataset = self.__explodedataframe(explodestruct, explodearray, dataset, tempview)

                if flatten:
                    dataset = self.flatten(dataset, select_columns, stop_at_column)

                if self.__tableprops["DEBUG"]:
                    Writer.logger(f"Read xml file from {filename}")
                    Writer.logger(f"Temp view created as {tempview}")
                    Writer.logger(f"Records => {dataset.count()}")
                    # dataset.printSchema()
                    Writer.logger(self.__getDFSchema(dataset))
                    # dataset.show(2, truncate=False, vertical=True)
                    Writer.logger(self.__getDFSamples(dataset))

                return dataset
            except Exception as e:
                Writer.logger("exception in loadxml")
                Writer.logger(e) 
        else:
            print(f"{TEST_NOT_INITIATED_MESSAGE} Project name < 3 chars & desc not set < 10 chars.")
            return None

    # load json file
    def loadjson(self, filename, tempview, useroptions=None, explodestruct=False, explodearray=False, flatten=False, select_columns=None, stop_at_column=None):
        if self.__check_run():
            try:
                Writer.logger("load json files...", True)
                jsondefaultoptions = {"multiline": "false"}

                if not useroptions:
                    dataset = spark.read.format("json").options(**jsondefaultoptions).load(filename) \
                        .withColumn(input_filename, col("_metadata.file_path"))
                else:
                    useroptions = self.__adddefaultoptions(useroptions, jsondefaultoptions)
                    dataset = spark.read.format("json").options(**useroptions).load(filename) \
                        .withColumn(input_filename, col("_metadata.file_path"))

                # dataset.createOrReplaceTempView(tempview)
                dataset.cache()
                self._settings.createTempViews(dataset, tempview)

                if tempview not in self.__src_trg_file_list:
                    self.__src_trg_file_list.append(tempview)

                if explodestruct or explodearray:
                    dataset = self.__explodedataframe(explodestruct, explodearray, dataset, tempview)

                if flatten:
                    dataset = self.flatten(dataset, select_columns, stop_at_column)

                if self.__tableprops["DEBUG"]:
                    Writer.logger(f"Read json file from {filename}")
                    Writer.logger(f"Temp view created as {tempview}")
                    Writer.logger(f"Records => {dataset.count()}")
                    # dataset.printSchema()
                    Writer.logger(self.__getDFSchema(dataset))
                    # dataset.show(2, truncate=False, vertical=True)
                    Writer.logger(self.__getDFSamples(dataset))

                return dataset
            except Exception as e:
                Writer.logger("exception in loadjson")
                Writer.logger(e)
        else:
            print(f"{TEST_NOT_INITIATED_MESSAGE} Project name < 3 chars & desc not set < 10 chars.")
            return None

    # load parquet file
    def loadparquet(self, filename, tempview, useroptions=None, explodestruct=False, explodearray=False, flatten=False, select_columns=None, stop_at_column=None):
        if self.__check_run():
            try:
                Writer.logger("load parquet files...", True)
                if not useroptions:
                    dataset = spark.read.format("parquet").load(filename) \
                        .withColumn(input_filename, col("_metadata.file_path"))
                else:
                    dataset = spark.read.format("parquet").options(**useroptions).load(filename) \
                        .withColumn(input_filename, col("_metadata.file_path"))

                # dataset.createOrReplaceTempView(tempview)
                dataset.cache()
                self._settings.createTempViews(dataset, tempview)

                if tempview not in self.__src_trg_file_list:
                    self.__src_trg_file_list.append(tempview)

                if explodestruct or explodearray:
                    dataset = self.__explodedataframe(explodestruct, explodearray, dataset, tempview)

                if flatten:
                    dataset = self.flatten(dataset, select_columns, stop_at_column)

                if self.__tableprops["DEBUG"]:
                    Writer.logger(f"Read parquet file from {filename}")
                    Writer.logger(f"Temp view created as {tempview}")
                    Writer.logger(f"Records => {dataset.count()}")
                    # dataset.printSchema()
                    Writer.logger(self.__getDFSchema(dataset))
                    # dataset.show(2, truncate=False, vertical=True)
                    Writer.logger(self.__getDFSamples(dataset))

                return dataset
            except Exception as e:
                Writer.logger("exception in loadparquet")
                Writer.logger(e)
        else:
            print(f"{TEST_NOT_INITIATED_MESSAGE} Project name < 3 chars & desc not set < 10 chars.")
            return None
        
    # load delta file
    def loaddelta(self, filename, tempview, useroptions=None, explodestruct=False, explodearray=False, flatten=False, select_columns=None, stop_at_column=None):
        if self.__check_run():
            try:
                Writer.logger("load delta files...", True)
                if not useroptions:
                    dataset = spark.read.format("delta").load(filename) \
                        .withColumn(input_filename, col("_metadata.file_path"))
                else:
                    dataset = spark.read.format("delta").options(**useroptions).load(filename) \
                        .withColumn(input_filename, col("_metadata.file_path"))

                # dataset.createOrReplaceTempView(tempview)
                dataset.cache()
                self._settings.createTempViews(dataset, tempview)

                if tempview not in self.__src_trg_file_list:
                    self.__src_trg_file_list.append(tempview)

                if explodestruct or explodearray:
                    dataset = self.__explodedataframe(explodestruct, explodearray, dataset, tempview)

                if flatten:
                    dataset = self.flatten(dataset, select_columns, stop_at_column)

                if self.__tableprops["DEBUG"]:
                    Writer.logger(f"Read delta file from {filename}")
                    Writer.logger(f"Temp view created as {tempview}")
                    Writer.logger(f"Records => {dataset.count()}")
                    # dataset.printSchema()
                    Writer.logger(self.__getDFSchema(dataset))
                    # dataset.show(2, truncate=False, vertical=True)
                    Writer.logger(self.__getDFSamples(dataset))

                return dataset
            except Exception as e:
                Writer.logger("exception in loaddelta")
                Writer.logger(e)
        else:
            print(f"{TEST_NOT_INITIATED_MESSAGE} Project name < 3 chars & desc not set < 10 chars.")
            return None
        
    # load hive table
    def loadhive(self, hivetable, tempview, explodestruct=False, explodearray=False, flatten=False, select_columns=None, stop_at_column=None):
        if self.__check_run():
            try:
                Writer.logger("load hive tables...", True)
                if ('select' in hivetable or 'from' in hivetable or 'with' in hivetable or
                        ' ' in hivetable or '\r' in hivetable or '\n' in hivetable):
                    dataset = self._settings.sprk.sql(hivetable)
                else:
                    dataset = self._settings.sprk.sql(f"select * from {hivetable}")

                # dataset.createOrReplaceTempView(tempview)
                dataset.cache()
                self._settings.createTempViews(dataset, tempview)

                if tempview not in self.__src_trg_file_list:
                    self.__src_trg_file_list.append(tempview)

                if explodestruct or explodearray:
                    dataset = self.__explodedataframe(explodestruct, explodearray, dataset, tempview)

                if flatten:
                    dataset = self.flatten(dataset, select_columns, stop_at_column)

                if self.__tableprops["DEBUG"]:
                    Writer.logger(f"Read hive table from {hivetable}")
                    Writer.logger(f"Temp view created as {tempview}")
                    Writer.logger(f"Records => {dataset.count()}")
                    # dataset.printSchema()
                    Writer.logger(self.__getDFSchema(dataset))
                    # dataset.show(2, truncate=False, vertical=True)
                    Writer.logger(self.__getDFSamples(dataset))

                return dataset
            except Exception as e:
                Writer.logger("exception in loadhive")
                Writer.logger(e)
        else:
            print(f"{TEST_NOT_INITIATED_MESSAGE} Project name < 3 chars & desc not set < 10 chars.")
            return None

    # load data from any jdbc
    #for oracle use useroptions={"query":"select * from table"}
    #select * from table where col='ddd' --> with tmp_ora as (select * from table where col='ddd') --> select * from tmp_ora
    #with tmp_ora as (select * from table where col='ddd') -> useroptions={"query":"select * from tmp_ora"}
    def loadrdbms(self, driver, jdbcuri, dbuser, dbpassword, dbtable, tempview, useroptions=None, explodestruct=False,
                  explodearray=False, flatten=False, select_columns=None, stop_at_column=None):
        if self.__check_run():
            try:
                Writer.logger("load rdbms tables...", True)
                if self._settings.vencrypted_password:
                    vdbpass = PasswordManager.decrypt(dbpassword)
                else:
                    vdbpass=dbpassword

                rdbmsdefaultoptions={}
                if " " not in str(dbtable).strip(): # check if no select/ statement is present
                    rdbmsdefaultoptions = {"url": jdbcuri, "dbtable": dbtable, "user": dbuser,
                                           "password": vdbpass, "driver": driver}
                else:
                    if "oracle.jdbc.driver.oracledriver" in str(driver).lower():# for oracle
                        if str(dbtable).lower().startswith("with"):
                            try:
                                prm_qry=useroptions.get("query")
                                if len(str(prm_qry))==0:
                                    prm_qry=None
                            except:
                                prm_qry=None

                            if prm_qry is not None:
                                rdbmsdefaultoptions = {"url": jdbcuri, "prepareQuery": dbtable, "query": prm_qry,
                                                   "user": dbuser,
                                                   "password": vdbpass, "driver": driver}
                            else:
                                Writer.logger("missing query in user options, set useroptions={'query':'sqlquery'}", True)
                                rdbmsdefaultoptions=None
                        else:
                            rdbmsdefaultoptions = {"url": jdbcuri, "prepareQuery": f"with tmp_ora as ({dbtable})", "query": "select * from tmp_ora",
                                                   "user": dbuser,
                                                   "password": vdbpass, "driver": driver}
                    else:# for other than oracle
                        try:
                            prm_qry = useroptions.get("query")
                            if len(str(prm_qry)) == 0:
                                prm_qry = None
                        except:
                            prm_qry = None

                        if prm_qry is None:
                            rdbmsdefaultoptions = {"url": jdbcuri, "prepareQuery": dbtable, "query": "select 1",
                                                   "user": dbuser,
                                                   "password": vdbpass, "driver": driver}
                        else:
                            rdbmsdefaultoptions = {"url": jdbcuri, "prepareQuery": dbtable, "query": prm_qry,
                                                   "user": dbuser,
                                                   "password": vdbpass, "driver": driver}

                if rdbmsdefaultoptions=={} or rdbmsdefaultoptions is None:
                    return None
                else:
                    if not useroptions:
                        dataset = spark.read.format("jdbc").options(**rdbmsdefaultoptions).load()
                    else:
                        useroptions = self.__adddefaultoptions(useroptions, rdbmsdefaultoptions)
                        dataset = spark.read.format("jdbc").options(**useroptions).load()

                    # dataset.createOrReplaceTempView(tempview)
                    dataset.cache()
                    self._settings.createTempViews(dataset, tempview)
                    if tempview not in self.__src_trg_file_list:
                        self.__src_trg_file_list.append(tempview)
                    if explodestruct or explodearray:
                        dataset = self.__explodedataframe(explodestruct, explodearray, dataset, tempview)
                    if flatten:
                        dataset = self.flatten(dataset, select_columns, stop_at_column)
                    if self.__tableprops["DEBUG"]:
                        Writer.logger(f"Read rdbms from {jdbcuri}")
                        Writer.logger(f"Temp view created as {tempview}")
                        Writer.logger(f"Records => {dataset.count()}")
                        # dataset.printSchema()
                        Writer.logger(self.__getDFSchema(dataset))
                        # dataset.show(2, truncate=False, vertical=True)
                        Writer.logger(self.__getDFSamples(dataset))
                    return dataset
            except Exception as e:
                Writer.logger("exception in loadrdbms")
                Writer.logger(e)
        else:
            print(f"{TEST_NOT_INITIATED_MESSAGE} Project name < 3 chars & desc not set < 10 chars.")
            return None

    # clear response collection
    def clearResponsecollection(self):
        self.__responsecollection = {}

    # load data from rest api
    def loadrestapi(self, method="GET", endpoint="", headers={}, data={}, jsondata={}, variables={}, timeout=10000, retry=1, tempview="", sql="", ignoressl=True,
                    repartition=False, explodestruct=False, explodearray=False, flatten=False, select_columns=None, stop_at_column=None):
        if self.__check_run():
            try:
                Writer.logger("load restapi...", True)
                #defHeaders = {"Content-Type": "application/json", "Accept": "application/json"}
                #headers = self.__adddefaultoptions(headers, defHeaders)
                dataset = None
                r = Rest()
                r.set_parameters("restapi", endpoint, method, data, jsondata, variables, self.__responsecollection, headers, timeout, retry, ignoressl)
                jsonresponse = r.getresponse()
                self.__responsecollection[tempview] = jsonresponse

                if len(sql.strip())>0:
                    # r.getdataframe(jsonresponse).createOrReplaceTempView(tempview)
                    self._settings.createTempViews(r.getdataframe(jsonresponse), tempview)
                    dataset = self._settings.sprk.sql(sql)
                    # dataset.createOrReplaceTempView(tempview)
                    dataset.cache()
                    self._settings.createTempViews(dataset, tempview)

                    if tempview not in self.__src_trg_file_list:
                        self.__src_trg_file_list.append(tempview)

                    if explodestruct or explodearray:
                        dataset = self.__explodedataframe(explodestruct, explodearray, dataset, tempview)

                    if flatten:
                        dataset = self.flatten(dataset, select_columns, stop_at_column)

                    self.__responsecollection = {}
                    return dataset
                else:
                    return jsonresponse


            except Exception as e:
                Writer.logger("exception in loadrestapi")
                Writer.logger(e)
        else:
            print(f"{TEST_NOT_INITIATED_MESSAGE} Project name < 3 chars & desc not set < 10 chars.")
            return None

    # load data from static sql
    def loadsql(self, sql, tempview):
        if self.__check_run():
            try:
                Writer.logger("load sql...", True)
                dataset = self._settings.sprk.sql(sql)

                # dataset.createOrReplaceTempView(tempview)
                dataset.cache()
                self._settings.createTempViews(dataset, tempview)

                if tempview not in self.__src_trg_file_list:
                    self.__src_trg_file_list.append(tempview)

                if self.__tableprops["DEBUG"]:
                    Writer.logger(f"Read rdbms from {sql}")
                    Writer.logger(f"Temp view created as {tempview}")
                    Writer.logger(f"Records => {dataset.count()}")
                    # dataset.printSchema()
                    Writer.logger(self.__getDFSchema(dataset))
                    Writer.logger(self.__getDFSamples(dataset))

                return dataset

            except Exception as e:
                Writer.logger("exception in loadsql")
                Writer.logger(e)
        else:
            print(f"{TEST_NOT_INITIATED_MESSAGE} Project name < 3 chars & desc not set < 10 chars.")
            return None

    def buildtable(self, tablename:str, sql:str, pk:str, qryType="") -> Dict[str, Any]:
        """
            Creates a tempview from the sql and creates a table from the tempview with two columns,
            PK_KEY (concatenated records from primary keys) and REC_COUNT (number of records in the group).
            Args:
                tablename (str): Name of the tempview to be created
                sql (str): SQL query to be executed to create tempview
                pk (str): Primary key columns separated by comma in case it's composed from more than one column
            Returns:
                {
                    "TABLENAME": Name of the table with primary key and count,
                    "PK": Primary key name,
                    "DF": SparkDataframe with primary key and count 
                }
        """
        print("In Build Table....")
        if self.__check_run():
            retval = {}
            try:
                Writer.logger("build source/target query...", True)
                pk = pk.strip()
                hasnopk = Atom_UDF.iif(pk == "", True, False)
                hascompositepk = ',' in pk
                pkkey = pkcols = allcols = groupedcols = ""

                df = self._settings.sprk.sql(sql)
                #df.createOrReplaceTempView(tablename)
                self._settings.createTempViews(df, tablename)
                tmp_tablename = f"tmp_{tablename}"
                df.cache()
                cols = df.columns
                df = df[sorted(cols)]
                v = -1

                if hasnopk:
                    retval = {"TABLENAME": tmp_tablename, "PKEY": "PK_KEY", "DF": None}
                    fields = [
                        field for field in df.schema.fields
                        if field.name.lower() not in self.__tableprops["EXCLUDECOLS-COLTEST"]
                    ]
                    for s in fields:
                        groupedcols = groupedcols + "`" + s.name.lower() + '`,'
                        simple = s.dataType.simpleString()
                        lower = simple.lower()
                        
                        dtypes1 = (
                            "string", "boolean", "numeric", "tinyint", "smallint",
                            "bigint", "int", "void", "integer",  "double"
                        )
                        dtypes2 = ("varchar", "char", "text", "float", "decimal", "doubleprecision")
                        if simple.lower() in dtypes1 or simple.startswith(dtypes2):
                            pkkey = pkkey + "nvl(trim(cast(`{0}` as string)),''),'|',".format(s.name)
                            pkcols = pkcols + s.name + ','
                            if pk.strip().lower() == s.name.lower():
                                allcols = allcols + "pk(nvl(trim(cast(`{0}` as string)),''), '{1}') as `{0}`,".format(
                                    s.name, self.__tableprops["CLEANPKLEVEL"]
                                )
                            else:
                                allcols = allcols + "cleantabledelimiters(nvl(cast(`{0}` as string),'')) as `{0}`,".format(
                                    s.name)
                        elif lower == 'date':
                            #pkkey = pkkey + "nvl(trim(cast(from_unixtime(unix_timestamp(`{0}`),'yyyy-MM-dd') as string)),''),'|',".format(
                            #    s.name)
                            pkkey = pkkey + "nvl(trim(cast(date_format(`{0}`, 'yyyy-MM-dd') as string)),''),'|',".format(
                                s.name)
                            pkcols = pkcols + s.name + ','

                            if pk.strip().lower() == s.name.lower():
                                #allcols = allcols + "pk(nvl(trim(cast(from_unixtime(unix_timestamp(`{0}`),'yyyy-MM-dd') as " \
                                #                    "string)),''),'{1}') as `{0}`,".format(
                                #    s.name, self.__tableprops["CLEANPKLEVEL"])
                                allcols = allcols + "pk(nvl(trim(cast(date_format(`{0}`,'yyyy-MM-dd') as " \
                                                    "string)),''),'{1}') as `{0}`,".format(
                                    s.name, self.__tableprops["CLEANPKLEVEL"])
                            else:
                                #allcols = allcols + "cleantabledelimiters(nvl(cast(from_unixtime(unix_timestamp(`{0}`)," \
                                #                    "'yyyy-MM-dd') as string),'')) as `{0}`,".format(s.name)
                                allcols = allcols + "cleantabledelimiters(nvl(cast(date_format(`{0}`," \
                                                    "'yyyy-MM-dd') as string),'')) as `{0}`,".format(s.name)
                        elif s.dataType.simpleString().lower() == 'timestamp':
                            #pkkey = pkkey + "nvl(trim(cast(from_unixtime(unix_timestamp(`{0}`),'yyyy-MM-dd HH:mm:ss') as " \
                            #                "string)),''),'|',".format(s.name)
                            pkkey = pkkey + "nvl(trim(cast(date_format(`{0}`,'yyyy-MM-dd HH:mm:ss.SSS') as " \
                                            "string)),''),'|',".format(s.name)
                            pkcols = pkcols + s.name + ','

                            if pk.strip().lower() == s.name.lower():
                                #allcols = allcols + "pk(nvl(trim(cast(from_unixtime(unix_timestamp(`{0}`),'yyyy-MM-dd " \
                                #                    "HH:mm:ss') as string)),''),'{1}') as `{0}`,".format(s.name,
                                #                                                                         self.__tableprops[
                                #                                                                             "CLEANPKLEVEL"])
                                allcols = allcols + "pk(nvl(trim(cast(date_format(`{0}`,'yyyy-MM-dd " \
                                                    "HH:mm:ss.SSS') as string)),''),'{1}') as `{0}`,".format(s.name,
                                                                                                            self.__tableprops[
                                                                                                                "CLEANPKLEVEL"])
                            else:
                                #allcols = allcols + "cleantabledelimiters(nvl(cast(from_unixtime(unix_timestamp(`{0}`)," \
                                #                    "'yyyy-MM-dd HH:mm:ss') as string),'')) as `{0}`,".format(
                                #    s.name)
                                allcols = allcols + "cleantabledelimiters(nvl(cast(date_format(`{0}`," \
                                                    "'yyyy-MM-dd HH:mm:ss') as string),'')) as `{0}`,".format(
                                    s.name)
                        elif s.dataType.simpleString().lower().startswith('array'):
                            pkkey = pkkey + "nvl(trim(cast(concat_ws(';', sort_array(cast(`{0}` as array<string>))) as string)),''),'|',".format(
                                s.name)
                            pkcols = pkcols + s.name + ','

                            if pk.strip().lower() == s.name.lower():
                                allcols = allcols + "pk(nvl(trim(cast(concat_ws(';', sort_array(cast(`{0}` as array<string>))) as string)),'')," \
                                                    "'{1}') as `{0}`,".format(
                                    s.name, self.__tableprops["CLEANPKLEVEL"])
                            else:
                                allcols = allcols + "cleantabledelimiters(nvl(cast(concat_ws(';', sort_array(cast(`{0}` as array<string>))) as " \
                                                    "string),'')) as `{0}`,".format(
                                    s.name)
                        else:
                            pkkey = pkkey + "nvl(trim(cast(hash(`{0}`) as string)),''),'|',".format(s.name)
                            pkcols = pkcols + s.name + ','
                            if pk.strip().lower() == s.name.lower():
                                allcols = allcols + "cleantabledelimiters(nvl(trim(cast(hash(pk(`{0}`, '{1}')) as string))," \
                                                    "'')) as `{0}`,".format(
                                    s.name, self.__tableprops["CLEANPKLEVEL"])
                            else:
                                allcols = allcols + "cleantabledelimiters(nvl(cast(hash(`{0}`) as string),'')) as `{0}`,".format(
                                    s.name)

                    col_order = pkcols[0:-1] #last char is ","
                    Writer.logger('Key Column order =>' + col_order)
                    del df
                    gc.collect()
                    #df = self._settings.sprk.sql(f"select count(1) as r_count from {tablename}")
                    #df.cache()
                    #v = df.first()[0]

                    #del df
                    #gc.collect()
                    Writer.logger(
                        "Test qry => select {0} from <<src/trg_table_name>> where concat({1})='<<value>>'".format(
                            col_order, col_order.replace(",", ", '|' ,")
                        )
                    )
                    tmp_qry = "select pk(concat({0}),'{4}') as PK_KEY, count(1) as REC_COUNT from {2} t group by {3}".format(
                        pkkey.strip()[0:-5],
                        allcols.strip()[0:-1],
                        tablename,
                        groupedcols.strip()[0:-1],
                        self.__tableprops["CLEANPKLEVEL"]
                    )
                    df = self._settings.sprk.sql(tmp_qry)
                    df.cache()
                elif hascompositepk:
                    retval = {"TABLENAME": tmp_tablename, "PKEY": "PK_KEY", "DF": None}
                    del df
                    gc.collect()

                    #Writer.logger("counting col using select", True)
                    #df = self._settings.sprk.sql(f"select count(1) as r_count from {tablename}")
                    #df.cache()
                    #v = df.first()[0]

                    #del df
                    #gc.collect()

                    df = self._settings.sprk.sql(
                        "select pk(trim(concat({0})),'{3}') as PK_KEY, {1} from {2} t".format(
                            pk, '*', tablename, self.__tableprops["CLEANPKLEVEL"]
                        )
                    )
                    df.cache()
                else:
                    retval = {"TABLENAME": tablename, "PKEY": pk, "DF": None}
                    del df
                    gc.collect()
                    #df = self._settings.sprk.sql("select count(1) as count from {1} t".format('*', tablename))
                    #df.cache()
                    #v = df.first()[0]

                    #del df
                    #gc.collect()

                    df = self._settings.sprk.sql("select {0} from {1} t".format('*', tablename))
                    df.cache()
                
                #print(f"count1==={v}")
                #print("create temp view")
                print("Create temp view")
                df.createOrReplaceTempView(tmp_tablename)
                #self._settings.createTempViews(df, tmp_tablename)

                retval["DF"] = df
                if self.__tableprops["DEBUG"]:
                    Writer.logger(f"{qryType}query ok...")
                    Writer.logger(f"Records => {df.count()}")
                    no_pk_msg = f"No key is given, created a composite key: {pkkey.strip()[0:-5]}"
                    Writer.logger(no_pk_msg if hasnopk else f"Given key is: {pk}")
                    # Writer.logger("**SQL**")
                    # Writer.logger(sql)
                    # Writer.logger("**end of sql**")
                    Writer.logger(self.__getDFSchema(df))
                    Writer.logger(self.__getDFSamples(df))
            except Exception as e:
                print(e)
                del df
                gc.collect()
                self._settings.cleartempviews()
                Writer.logger("exception in buildtable", True)
                self._settings.sprk.sql(f"drop view if exists {tablename}")
                Writer.logger(e)
                retval = {}
        
        if not retval:
            print(f"{TEST_NOT_INITIATED_MESSAGE} Project name < 3 chars & desc not set < 10 chars.")
        return retval

    def comparecolumns(self, srctablename:str, trgtablename:str):
        if self.__check_run():
            try:
                if not (self.__resdetailsummary['COL_TEST']['START_STATUS'] and
                        self.__resdetailsummary['COL_TEST']['END_STATUS']):
                    comp_cols_onfiles = (srctablename in self.__src_trg_file_list) and (trgtablename in self.__src_trg_file_list)
                    if comp_cols_onfiles:
                        flag_str = "on files"
                        self.__resdetailsummary['COL_TEST']['TEST_TYPE']="F"
                    else:
                        flag_str = "on query"
                        self.__resdetailsummary['COL_TEST']['TEST_TYPE'] = "Q"

                    sdf = None
                    tdf = None
                    coltestres = None
                    Writer.logger(f"starting column test... {flag_str}", True)
                    if self.__tableprops["COLTEST"]:
                        self.__resdetailsummary['COL_TEST']['START_STATUS'] = True
                        self.__resdetailsummary['COL_TEST']['START_TIME'] = Atom_UDF.gettimestamp()
                        excollist = "','".join(self.__tableprops["EXCLUDECOLS-COLTEST"])
                        datatypesql = ''

                        if self.__tableprops["COLTEST-DATATYPE"]:
                            datatypesql = "and lower(st.data_type)=lower(tt.data_type)"

                        sdf = self._settings.sprk.sql(f"desc {srctablename}")
                        sdf.cache()
                        # srcCols = sdf.count()
                        self.__resdetailsummary['COL_TEST']['SRC_COL_COUNT'] = sdf.count()
                        #if comp_cols_onfiles:
                        self.__ressummary['MD_SRC_COL_COUNT'] = self.__ressummary['MD_SRC_COL_COUNT'] + \
                                                                self.__resdetailsummary['COL_TEST']['SRC_COL_COUNT']
                        # sdf.createOrReplaceTempView(f"{srctablename}_ct")
                        self._settings.createTempViews(sdf, f"{srctablename}_ct")

                        tdf = self._settings.sprk.sql(f"desc {trgtablename}")
                        tdf.cache()
                        # trgCols = tdf.count()
                        self.__resdetailsummary['COL_TEST']['TRG_COL_COUNT'] = tdf.count()
                        #if comp_cols_onfiles:
                        self.__ressummary['MD_TRG_COL_COUNT'] = self.__ressummary['MD_TRG_COL_COUNT'] + \
                                                                self.__resdetailsummary['COL_TEST']['TRG_COL_COUNT']
                        # tdf.createOrReplaceTempView(f"{trgtablename}_ct")
                        self._settings.createTempViews(tdf, f"{trgtablename}_ct")

                        if len(self.__tableprops["EXCLUDECOLS-COLTEST"]) > 0:
                            sdf = self._settings.sprk.sql(
                                f"select * from {srctablename}_ct where lower(col_name) not in ('{excollist}')")
                            # sdf.createOrReplaceTempView(f"{srctablename}_ct")
                            self.__resdetailsummary['COL_TEST']['SRC_COL_COUNT'] = sdf.count()
                            sdf.cache()
                            self._settings.createTempViews(sdf, f"{srctablename}_ct")

                            tdf = self._settings.sprk.sql(
                                f"select * from {trgtablename}_ct where lower(col_name) not in ('{excollist}')")
                            # tdf.createOrReplaceTempView(f"{trgtablename}_ct")
                            self.__resdetailsummary['COL_TEST']['TRG_COL_COUNT'] = tdf.count()
                            tdf.cache()
                            self._settings.createTempViews(tdf, f"{trgtablename}_ct")

                        del sdf
                        del tdf
                        gc.collect()

                        coltestsql = "with col_test as (select lower(st.col_name) as src_col_name, lower(st.data_type) " \
                                     "as src_data_type, lower(tt.col_name) as trg_col_name, lower(tt.data_type) as \n " \
                                     f"trg_data_type from {srctablename}_ct st full outer join {trgtablename}_ct tt " \
                                     f"on(lower(st.col_name)=lower(tt.col_name) " \
                                     f"{datatypesql})) \n select src_col_name as col_name, src_data_type as data_type, " \
                                     "'Not Found in Target' as error_desc " \
                                     "from col_test where (trg_col_name is null or trg_data_type is null)  \n " \
                                     " union all \n select trg_col_name as col_name, trg_data_type as data_type, " \
                                     "'Not Found in Source' as error_desc from col_test where " \
                                     "(src_col_name is null or src_data_type is null)\n"

                        coltestres = self._settings.sprk.sql(coltestsql)
                        coltestres.cache()
                        coltesterrors = coltestres.count()
                        # coltesterrors = coltesterrors the nbloe code is not requireed but leaving it as is
                        self.__resdetailsummary['COL_TEST']['COLTEST_ERRORS'] = \
                            self.__resdetailsummary['COL_TEST']['COLTEST_ERRORS'] + coltesterrors

                        self.__resdetailsummary['COL_TEST']['CUR_COLTEST_ERRORS'] = coltesterrors

                        self.__ressummary['COLTEST_ERRORS'] = self.__ressummary['COLTEST_ERRORS'] + coltesterrors

                        if coltesterrors > 0:
                            Writer.logger("writing logs...", True)
                            Writer.qa_metadatacomparison(coltestres, self.__tableprops)
                            Writer.logger(f"\t{coltesterrors} records added.", True)
                            Writer.logger("writing logs...completed", True)

                            if self.__tableprops["DEBUG"]:
                                Writer.logger("Columns Test")
                                Writer.logger(f"{coltesterrors} mismatches in column test")
                                # coltestres.show(2, truncate=False, vertical=True)
                                Writer.logger(self.__getDFSamples(coltestres))
                        else:
                            Writer.logger("No mismatches in column test", True)


                        Writer.logger("column test completed", True)

                        # drop the tempviews
                        self._settings.cleartempview(f"{srctablename}_ct")
                        self._settings.cleartempview(f"{trgtablename}_ct")


                        self.__resdetailsummary['COL_TEST']['END_STATUS'] = True
                        et = Atom_UDF.gettimestamp()
                        self.__resdetailsummary['COL_TEST']['END_TIME'] = et
                        self.__resdetailsummary['END_TIME'] = et
                    else:
                        Writer.logger("skpping column test")

                    #del coltestres
                    #del sdf
                    #del tdf
                    #gc.collect()
                    return self.__resdetailsummary['COL_TEST']
                else:
                    print("col test is already completed.")
                    return self.__resdetailsummary['COL_TEST']
            except Exception as e:
                Writer.logger("exception in comparecolumns", True)
                Writer.logger(e, True)
                self._settings.cleartempview(f"{srctablename}_ct")
                self._settings.cleartempview(f"{trgtablename}_ct")

                del coltestres
                del sdf
                del tdf
                gc.collect()

                self._settings.cleartempview(f"{srctablename}_ct")
                self._settings.cleartempview(f"{trgtablename}_ct")

                self.__resdetailsummary['COL_TEST']['END_STATUS'] = False
                self.__resdetailsummary['COL_TEST']['EXCEPTION'] = str(e)
                # drop the tempviews
                self._settings.sprk.sql(f"drop view if exists {srctablename}_ct")
                self._settings.sprk.sql(f"drop view if exists {trgtablename}_ct")
                et = Atom_UDF.gettimestamp()
                self.__resdetailsummary['COL_TEST']['END_TIME'] = et
                self.__resdetailsummary['END_TIME'] = et
                return self.__resdetailsummary['COL_TEST']
        else:
            print(f"{TEST_NOT_INITIATED_MESSAGE} Project name < 3 chars & desc not set < 10 chars.")
            return None

    def comparekeycolumn(self, src:Dict[str, Any], trg:Dict[str, Any]):
        def get_errors_write_key_comparison(
                tableprops:Dict[str, Any], key_query,
                test_type:Literal["unmatching records", "duplicates in source", "duplicates in target"]
            ) -> int:
            self._settings.setsparkobject(SparkSession.builder.getOrCreate())
            print(f"Test type: {test_type}\nCreating keyres")
            keyres = self._settings.sprk.sql(key_query)
            keyres.cache()

            print("Counting key errors")
            keyerrors = keyres.count()
            if keyerrors:
                Writer.logger("writing logs...", True)
                Writer.qa_keycomparison(keyres, tableprops)
                Writer.logger(f"\t{keyerrors} records added.", True)
                Writer.logger("writing logs...completed", True)

                get_iif = lambda dct: Atom_UDF.iif(dct["PKEY"] == '', 'PK_KEY', dct["PKEY"])
                if tableprops["DEBUG"]:
                    Writer.logger(self.__getDFSamples(keyres))
                    Writer.logger("Source: {0}.{1}".format(src["TABLENAME"], get_iif(src)), True)
                    Writer.logger("Target: {0}.{1}".format(trg["TABLENAME"], get_iif(trg)), True)
            else:
                Writer.logger(f"No mismatches in key test for {test_type}", True)
            return keyerrors
        
        def set_details_summary(d_test_st, d_test_ss:bool, d_test_es:bool, dct:Dict[str, Any], msg:str = None):
            dct['KEY_TEST']['END_STATUS'] = msg is not None
            dct['DATA_TEST']['START_TIME'] = d_test_st
            dct['DATA_TEST']['START_STATUS'] = d_test_ss
            dct['DATA_TEST']['END_STATUS'] = d_test_es

            ts = Atom_UDF.gettimestamp()
            dct['KEY_TEST']['END_TIME'] = ts
            dct['DATA_TEST']['END_TIME'] = ts

            if msg:
                dct['KEY_TEST']['EXCEPTION'] = msg
            return dct
        
        if not self.__check_run():
            print(f"{TEST_NOT_INITIATED_MESSAGE} Project name < 3 chars & desc not set < 10 chars.")
            return None
        try:
            if self.__resdetailsummary['KEY_TEST']['START_STATUS'] and self.__resdetailsummary['KEY_TEST']['END_STATUS']:
                print("key test is already completed.")
            
            Writer.logger("testing key column...", True)
            self.__resdetailsummary['KEY_TEST']['START_STATUS'] = True
            keytest_start_time = Atom_UDF.gettimestamp()
            self.__resdetailsummary['KEY_TEST']['START_TIME'] = keytest_start_time
            Writer.logger("checking count src vs target...", True)

            src["DF"].cache()
            trg["DF"].cache()

            sdc = src["DF"].count()
            Writer.logger(f"source count {sdc}")
            tdc = trg["DF"].count()
            Writer.logger(f"target count {tdc}")

            if sdc > 0 and tdc > 0:
                # if data set has records
                Writer.logger("starting key test...", True)

                kt_notfound = "with a as (select distinct cast(nvl(st.{0}, '') as string) as source_table_key, " \
                                "cast(nvl(tt.{2},'') as string) as target_table_key " \
                                "from {1} st full join {3} tt on st.{0}=tt.{2} " \
                                "where st.{0} is null or tt.{2} is null), " \
                                "srcvstrg as (select (case when source_table_key='' then target_table_key " \
                                "when target_table_key='' then source_table_key else '' end) as keycol, " \
                                "(case  when source_table_key='' then 'NOT FOUND IN SOURCE TABLE' " \
                                "when target_table_key='' then 'NOT FOUND IN TARGET TABLE' else '' end) as errortype " \
                                "from a where a.source_table_key!=a.target_table_key) select * from srcvstrg" \
                    .format(src["PKEY"], src["TABLENAME"], trg["PKEY"], trg["TABLENAME"])

                kt_dupsinsrc = "with dupsinsrc as (select cast(st.{0} as string) as keycol, " \
                                "'DUPLICATES IN SOURCE' as errortype " \
                                "from {1} st where st.{0} is not null " \
                                "group by st.{0} having count(st.{0})!=1) select * from dupsinsrc" \
                    .format(src["PKEY"], src["TABLENAME"])

                kt_dupsintrg = "with dupsintarg as (select cast(tt.{0} as string) as keycol, " \
                                "'DUPLICATES IN TARGET' as errortype " \
                                "from {1} tt where tt.{0} is not null " \
                                "group by tt.{0} having count(tt.{0})!=1) select * from dupsintarg" \
                    .format(trg["PKEY"], trg["TABLENAME"])

                keymatchessql = f"select count(1) as match_count from {src["TABLENAME"]} st, {trg["TABLENAME"]} tt "
                keymatchessql += f"where st.{src["PKEY"]}=tt.{trg["PKEY"]}"

                try:
                    Writer.logger("checking for matches...", True)
                    keyres = self._settings.sprk.sql(keymatchessql)

                    print("Keyres",keyres.first()[0])
                    keyres.cache()
                    keymatches = keyres.first()[0]
                    self.__resdetailsummary['KEY_TEST']['MATCH_FOUND'] = keymatches
                    del keyres
                    gc.collect()
                    Writer.logger(f"Total Matches= {keymatches}", True)

                    try:
                        Writer.logger("testing key column for unmatching records in source vs target...", True)
                        keyerrortotal = get_errors_write_key_comparison(self.__tableprops, kt_notfound, "unmatching records")

                        Writer.logger("testing key column for duplicates in source...", True)
                        keyerrortotal += get_errors_write_key_comparison(self.__tableprops, kt_dupsinsrc, "duplicates in source")

                        Writer.logger("testing key column for duplicates in target...")
                        keyerrortotal += get_errors_write_key_comparison(self.__tableprops, kt_dupsintrg, "duplicates in target")

                        Writer.logger(f"total key errors {keyerrortotal}", True)

                    except Exception as e:
                        if "DF" in src.keys():
                            del src["DF"]
                        if "DF" in trg.keys():
                            del trg["DF"]
                        del keyres
                        gc.collect()
                        Writer.logger("exception in comparekeycolumn (1)", True)
                        Writer.logger(e)
                        keyerrors = 0

                    self.__resdetailsummary['KEY_TEST']['KEYTEST_ERRORS'] += keyerrortotal
                    self.__ressummary['KEYTEST_ERRORS'] += keyerrortotal
                    Writer.logger("key test completed successfully", True)
                    self.__resdetailsummary['KEY_TEST']['END_STATUS'] = True
                    self.__resdetailsummary['KEY_TEST']['END_TIME'] = Atom_UDF.gettimestamp()
                    self.__resdetailsummary['END_TIME'] = Atom_UDF.gettimestamp()

                except Exception as e:
                    keyerrors = 0
                    keyerrortotal = 0
                    if "DF" in src.keys():
                        del src["DF"]
                    if "DF" in trg.keys():
                        del trg["DF"]
                    del keyres
                    gc.collect()
                    Writer.logger("exception in comparekeycolumn (2)", True)
                    Writer.logger(e)
                    self.__resdetailsummary['KEY_TEST']['MATCH_FOUND'] = 0
                    self.__resdetailsummary['KEY_TEST'] = False
                    self.__resdetailsummary['KEY_TEST']['EXCEPTION'] = str(e)
                    
                    et = Atom_UDF.gettimestamp()
                    self.__resdetailsummary['KEY_TEST']['END_TIME'] = et
                    self.__resdetailsummary['END_TIME'] = et

            elif sdc==0 and tdc==0:
                get_msg = lambda status: f"No records in source and target, but col. test {status}. (cannot verify)"
                if self.__tableprops["COLTEST"]:
                    tp_col_test = self.__resdetailsummary['COL_TEST']
                    # col test is set to true and status is pass
                    if (tp_col_test['END_STATUS'] and tp_col_test['CUR_COLTEST_ERRORS']==0):
                        Writer.logger("No records in source and target, but col. test passed.", True)
                        self.__resdetailsummary['DATA_TEST']['SRC_COL_COUNT'] = tp_col_test['SRC_COL_COUNT']
                        self.__resdetailsummary['DATA_TEST']['TRG_COL_COUNT']= tp_col_test['TRG_COL_COUNT']
                        self.__resdetailsummary = set_details_summary(keytest_start_time, True, True, self.__resdetailsummary)
                    else:
                        # col test is set to true and status is fail
                        Writer.logger("No records in source and target, but col. test failed.", True)
                        self.__resdetailsummary = set_details_summary(keytest_start_time, False, False, self.__resdetailsummary, get_msg("failed"))
                else:
                    # col test is set to false
                    Writer.logger("No records in source and target, but col. test not done.", True)
                    self.__resdetailsummary = set_details_summary(keytest_start_time, False, False, self.__resdetailsummary, get_msg("not done"))
                self.__resdetailsummary['END_TIME'] = Atom_UDF.gettimestamp()
            else:
                if "DF" in src.keys():
                    del src["DF"]
                if "DF" in trg.keys():
                    del trg["DF"]
                gc.collect()
                Writer.logger(f"cannot perform key test", True)
                Writer.logger(f"{sdc} records found in source, {tdc} records in target.", True)

                self.__resdetailsummary['KEY_TEST']['END_STATUS'] = True
                self.__resdetailsummary['KEY_TEST']['END_TIME'] = self.__resdetailsummary['KEY_TEST']['START_TIME']
                self.__resdetailsummary['END_TIME'] = Atom_UDF.gettimestamp()

        except Exception as e:
            if "DF" in src.keys():
                del src["DF"]
            if "DF" in trg.keys():
                del trg["DF"]
            gc.collect()
            Writer.logger("exception in comparekeycolumn", True)
            Writer.logger(e)
            self.__resdetailsummary['END_TIME'] = Atom_UDF.gettimestamp()
        return self.__resdetailsummary['KEY_TEST']

    def comparedata(self, src:Dict[str, Any], trg:Dict[str, Any]):
        if not self.__check_run():
            print(f"{TEST_NOT_INITIATED_MESSAGE} Project name < 3 chars & desc not set < 10 chars.")
            return None
        
        data_test = self.__resdetailsummary['DATA_TEST']
        try:
            if data_test['START_STATUS'] or data_test['END_STATUS']:
                print("data test is already completed.")
                return self.__resdetailsummary['DATA_TEST']
            
            Writer.logger("comparing data...", True)
            data_test['START_STATUS'] = True
            data_test['START_TIME'] = Atom_UDF.gettimestamp()
            data_test['FAILED_COLS'] = 0

            sqcode = tqcode = ''
            src["DF"].cache()
            sdc = 0 if src is None else src["DF"].count()

            trg["DF"].cache()
            tdc = 0 if trg is None else trg["DF"].count()

            Writer.logger(f"source count {sdc}", True)
            Writer.logger(f"target count {tdc}", True)

            stablename = Atom_UDF.iif(src["TABLENAME"] == '', 'defsrc', src["TABLENAME"])
            ttablename = Atom_UDF.iif(trg["TABLENAME"] == '', 'deftrg', trg["TABLENAME"])

            exclude_cols = self.__tableprops["EXCLUDECOLS-DATATEST"]
            get_count = lambda d: len(d["DF"].columns) - len(Atom_UDF.getCommonValues(d["DF"].columns, exclude_cols))
            cols = Atom_UDF.getCommonValues(src["DF"].columns, trg["DF"].columns)
            if len(cols)>1:
                data_test['SRC_COL_COUNT'] = get_count(src)
                data_test['TRG_COL_COUNT'] = get_count(trg)
                if sdc > 0 and tdc > 0:
                    Writer.logger('starting data test...', True)
                    # debug = self.__tableprops["DEBUG"]
                    spk = str(src["PKEY"]).lower()
                    tpk = str(trg["PKEY"]).lower()
                    compop = Atom_UDF.iif(self.__tableprops["REVERSECHK"], "", "!")

                    if len(cols) > 0:
                        data_test['SRC_REC_COUNT'] = sdc
                        data_test['TRG_REC_COUNT'] = tdc
                        self.__ressummary['DT_SRC_COL_COUNT'] += data_test['SRC_COL_COUNT']
                        self.__ressummary['SRC_REC_COUNT'] += data_test['SRC_REC_COUNT']
                        self.__ressummary['DT_TRG_COL_COUNT'] += data_test['TRG_COL_COUNT']
                        self.__ressummary['TRG_REC_COUNT'] += data_test['TRG_REC_COUNT']

                        datatesttql = datatesttql_all = ""
                        src_df_schema = Atom_UDF.getdfschema(src["DF"])
                        trg_df_schema = Atom_UDF.getdfschema(trg["DF"])

                        def get_qcode_date(col:str, dtype:Literal["date", "timestamp","void"], table:Literal["src", "trg"]) -> str:
                            col_name = "source_data" if table=="src" else "target_data"
                            alias   = "st" if table=="src" else "tt"
                            dtformat = "yyyy-MM-dd" if dtype == "date" else "yyyy-MM-dd HH:mm:ss.SSS"

                            if dtype in ("date", "timestamp"):
                                return f"nvl(cast(date_format({alias}.`{col}`, '{dtformat}') as string),'') as {col_name}"
                            if dtype == "void":
                                return f"nvl(cast({alias}.`{col}` as string),'') as {col_name}"
                            qcode = f"(case when isdate({alias}.`{col}`) then nvl(cast(date_format({alias}.`{col}`,"
                            qcode += f"'yyyy-MM-dd HH:mm:ss.SSS') as string),'') else nvl(cast({alias}.`{col}` as string), '') "
                            return qcode + f"end ) as {col_name}"
                        
                        def get_qcode_arr(col:str, dtype:str, table:Literal["sre", "trg"]):
                            col_name = "source_data" if table=="src" else "target_data"
                            alias    = f"st.`{col}`" if table=="src" else f"tt.`{col}`"
                            if dtype.startswith("array"):
                                return f"nvl(cast(concat_ws(';', sort_array(cast({alias} as array<string>))) as string),'') as {col_name}"
                            if dtype == "void":
                                return f"nvl(cast({alias} as string),'') as {col_name}"
                            return f"nvl(cast(concat_ws(';', sort_array(split({alias}, ';'))) as string),'') as {col_name}"
                        
                        def get_data_query(col:str, sqcode:str, tqcode:str, src_pk:str, trg_pk:str, str_tbl:str, trg_tbl:str):
                            inner = f"select '{col}' as field_name, st.{spk} as keycol,  {sqcode}, {tqcode} "
                            inner += f"from {stablename} st, {ttablename} tt "
                            inner += f"where nvl(cast(st.{spk} as string),'') = nvl(cast(tt.{tpk} as string),'')"
                            return f"select * from ({inner}) `tbl_{col}` where source_data {compop}= target_data\n union all \n"
                        
                        for col in cols:
                            if col not in (spk, tpk) and col not in exclude_cols:
                                #sdatatypestring = src["DF"].schema[col].dataType.simpleString().lower()
                                #tdatatypestring = trg["DF"].schema[col].dataType.simpleString().lower()

                                sdatatypestring = src_df_schema[col]["col_datatype"]
                                tdatatypestring = trg_df_schema[col]["col_datatype"]
                                
                                dtypes = ('string', 'boolean', 'numeric', 'tinyint', 'smallint', 'bigint', 'int', 'integer', 'double', 'void')
                                date_types = ('date', 'timestamp', 'void')
                                starts_with_dtypes = ('varchar', 'char', 'text', 'float', 'decimal', 'doubleprecision')
                                
                                long_cond = lambda s: (s in dtypes or s.startswith(starts_with_dtypes))
                                arr_cond = lambda s: s.startswith('array') or s == "void"
                                if long_cond(sdatatypestring) and long_cond(tdatatypestring):
                                    sqcode = f"nvl(cast(st.`{col}` as string),'') as source_data"
                                    tqcode = f"nvl(cast(tt.`{col}` as string),'') as target_data"
                                elif sdatatypestring in date_types or tdatatypestring in date_types:
                                    sqcode = get_qcode_date(col, sdatatypestring, "src")
                                    tqcode = get_qcode_date(col, tdatatypestring, "trg")
                                elif arr_cond(sdatatypestring) or arr_cond(tdatatypestring):
                                    sqcode = get_qcode_arr(col, sdatatypestring, "src")
                                    tqcode = get_qcode_arr(col, tdatatypestring, "trg")
                                else:
                                    sqcode = f"nvl(cast(hash(st.`{col}`) as string),'') as source_data"
                                    tqcode = f"nvl(cast(hash(tt.`{col}`) as string),'') as target_data"
                                datatesttql = get_data_query(col, sqcode, tqcode, spk, tpk, stablename, ttablename)

                                if (self._settings.runallcols):
                                    datatesttql_all += datatesttql
                                else:
                                    Writer.logger(f"...checking column '{col}'", True)
                                    dtest = self._settings.sprk.sql(datatesttql[0:-12].strip())
                                    dtest.cache()
                                    datamismatches = dtest.count()

                                    data_test['DATATEST_ERRORS'] += datamismatches
                                    self.__ressummary['DATATEST_ERRORS'] += datamismatches

                                    if datamismatches > 0:
                                        Writer.logger(f"writing logs for '{col}'...", True)
                                        Writer.logger(f"\t {datamismatches} mismatches found in '{col}'.", True)
                                        data_test['FAILED_COLS'] +=1

                                        Writer.qa_datacomparison(dtest, self.__tableprops)
                                        if self.__tableprops["DEBUG"]:
                                            Writer.logger(self.__getDFSamples(dtest))
                                    else:
                                        Writer.logger(f"\t NO mismatches found in '{col}'.", True)

                                    #del dtest
                                    #gc.collect()

                        if (self._settings.runallcols):
                            Writer.logger(f"...checking table {self.__tableprops['TABLENAME']}", True)
                            dtest = self._settings.sprk.sql(datatesttql_all[0:-12].strip())
                            dtest.cache()
                            datamismatches = dtest.count()
                            data_test['DATATEST_ERRORS'] = data_test['DATATEST_ERRORS'] + datamismatches

                            self.__ressummary['DATATEST_ERRORS'] = self.__ressummary['DATATEST_ERRORS'] + datamismatches
                            if datamismatches > 0:
                                Writer.logger(f"writing logs for '{self.__tableprops['TABLENAME']}'...", True)
                                Writer.logger(f"\t {datamismatches} mismatches found in '{self.__tableprops['TABLENAME']}'.", True)

                                dtest.createOrReplaceTempView("f_cols")
                                #dtest.show()
                                failedCols_count= self._settings.sprk.sql("select count(distinct field_name) from f_cols").first()[0]
                                data_test['FAILED_COLS'] =failedCols_count
                                self._settings.sprk.sql(f"drop view if exists f_cols")

                                Writer.qa_datacomparison(dtest, self.__tableprops)
                                if self.__tableprops["DEBUG"]:
                                    # dtest.show(2, truncate=False, vertical=True)
                                    Writer.logger(self.__getDFSamples(dtest))
                            else:
                                Writer.logger(f"\t NO mismatches found in '{self.__tableprops['TABLENAME']}'.", True)

                            #del dtest
                            #gc.collect()

                        Writer.logger("data test completed.", True)
                        if (data_test['SRC_COL_COUNT'] != data_test['TRG_COL_COUNT']):
                            Writer.logger("pair query has column mismatch, orphan columns ignored.", True)

                        data_test['END_STATUS'] = True
                        et = Atom_UDF.gettimestamp()
                        data_test['END_TIME'] = et
                        self.__resdetailsummary['END_TIME'] = et
                    else:
                        Writer.logger("no common cols in source & target.", True)
                        data_test['END_STATUS'] = False
                        data_test['EXCEPTION'] = 'no common cols in source & target.'
                        et = Atom_UDF.gettimestamp()
                        data_test['END_TIME'] = et
                        self.__resdetailsummary['END_TIME'] = et
                elif (sdc == 0 and tdc==0):
                    if self.__tableprops["COLTEST"]:
                        if (self.__resdetailsummary['COL_TEST']['END_STATUS'] and
                                self.__resdetailsummary['COL_TEST']['CUR_COLTEST_ERRORS']==0):
                            Writer.logger("No records in source and target, but col. test passed.", True)
                            data_test['END_STATUS'] = True
                            data_test['END_TIME'] = Atom_UDF.gettimestamp()
                        else:
                            Writer.logger("No records in source and target, but col. test failed.", True)
                            data_test['END_STATUS'] = False
                            data_test['END_TIME'] = Atom_UDF.gettimestamp()
                            data_test['EXCEPTION']= "No records in source and target, but col. test failed. (cannot verify)"
                    else:
                        Writer.logger("No records in source and target, but col. test not done.", True)
                        data_test['END_STATUS'] = False
                        data_test['END_TIME'] = Atom_UDF.gettimestamp()
                        data_test['EXCEPTION'] = "No records in source and target, but col. test not done. (cannot verify)"
                    self.__resdetailsummary['END_TIME'] = Atom_UDF.gettimestamp()
                else:
                    if "DF" in src.keys():
                        del src["DF"]
                    if "DF" in trg.keys():
                        del trg["DF"]
                    gc.collect()
                    Writer.logger("cannot perform data comparision", True)
                    Writer.logger(f"{sdc} records found in source, {tdc} records in target.", True)
                    data_test['SRC_REC_COUNT'] = sdc  # sdf.count()
                    data_test['TRG_REC_COUNT'] = tdc  # tdf.count()
                    data_test['END_STATUS'] = False
                    data_test['END_TIME'] = data_test['START_TIME']
                    self.__resdetailsummary['END_TIME'] = Atom_UDF.gettimestamp()
            else:
                Writer.logger("cannot perform data comparision as no common cols found.", True)
                Writer.logger("src cols==>", src["DF"].columns)
                Writer.logger("trg cols==>", trg["DF"].columns)
                Writer.logger("common cols==>", cols)
                data_test['END_STATUS'] = False
                data_test['EXCEPTION'] = "cannot perform data comparision as no common cols found."
                et = Atom_UDF.gettimestamp()
                data_test['END_TIME'] = et
                self.__resdetailsummary['END_TIME'] = et

        except Exception as e:
            Writer.logger("exception in comparedata", True)
            print(e)
            Writer.logger(e)
            del dtest
            if "DF" in src.keys():
                del src["DF"]
            if "DF" in trg.keys():
                del trg["DF"]
            gc.collect()
            data_test['END_STATUS'] = False
            data_test['EXCEPTION'] = str(e)
            et = Atom_UDF.gettimestamp()
            data_test['END_TIME'] = et
            self.__resdetailsummary['END_TIME'] = et
        self.__resdetailsummary['DATA_TEST'] = data_test
        return data_test 

    def dataquality(self, dfortempview=None, pkey="", columns="", dqname="", condition=""):
        if self.__check_run():
            if "," in pkey:
                pkey=f"concat({pkey})"

            if "," in columns:
                columns=f"concat({columns})"


            Writer.logger(f"data quality checks... {dqname}", True)
            resdataquality = {'COLNAME': '',
                              'COLNAME2': '',
                              'TABLENAME': '',
                              'TYPE': '',
                              'CONDITION': '',
                              'START_STATUS': False,
                              'START_TIME': '',
                              'END_TIME': '',
                              'DATAQUALITY_ERRORS': 0,
                              'LOG_PATH': '',
                              'END_STATUS': False,
                              'EXCEPTION': ''}

            if self.__ressummary['DATAQUALITY_ERRORS'] == -1:
                self.__ressummary['DATAQUALITY_ERRORS'] = 0

            if self.__ressummary['DATAQUALITY_TESTS'] == -1:
                self.__ressummary['DATAQUALITY_TESTS'] = 0
                self.__ressummary['DATAQUALITY_TEST_PASSED'] = 0
                self.__ressummary['DATAQUALITY_TEST_FAILED'] = 0

            self.__ressummary['DATAQUALITY_TESTS'] += 1

            try:

                self.__dqId = self.__dqId + 1
                resdataquality['COLNAME'] = dqname
                resdataquality['CONDITION'] = condition
                resdataquality['START_STATUS'] = True
                resdataquality['START_TIME'] = Atom_UDF.gettimestamp()

                tempView = ""
                if isinstance(dfortempview, DataFrame):
                    tempView = f"{dqname}_df"
                    try:
                        # dfortempview.createOrReplaceTempView(tempView)
                        self._settings.createTempViews(dfortempview, tempView)
                    except Exception as e:
                        # print("bad column name.")
                        tempView = "_df"
                        # dfortempview.createOrReplaceTempView(tempView)
                        self._settings.createTempViews(dfortempview, tempView)
                else:
                    tempView = str(dfortempview)

                resdataquality['TABLENAME'] = tempView

                if self.__tableprops["DEBUG"]:
                    Writer.logger("print parameters")
                    Writer.logger(f"PK_KEY='{pkey}'")
                    Writer.logger(f"Columns='{columns}'")
                    Writer.logger(f"Name='{dqname}'")
                    Writer.logger(f"Condition='{condition}'")


                if len(columns.strip()) > 0 and len(pkey.strip())>0 and len(condition.strip())>0 and len(dqname)>0:
                    tmp=condition.replace("'", "''")
                    try:
                        dqTest = self._settings.sprk.sql(f"select {pkey} as pkey, '{columns}' as failed_col, '{tmp}' as condition, "
                                               f"{columns} as failed_colvalue from {tempView} where not ({condition})")
                        dqTest.cache()
                        dqErrorCount = dqTest.count()
                        resdataquality['DATAQUALITY_ERRORS'] = dqErrorCount
                        self.__ressummary['DATAQUALITY_ERRORS'] += dqErrorCount
                        if dqErrorCount > 0:
                            tbl_name = self.__tableprops["TABLENAME"]
                            dq_name = dqname.replace(",", "_")

                            if dqname in self.__dqcols:
                                dq_name += f"_{self.__dqId}"
                                resdataquality['COLNAME2'] = dq_name
                                resdataquality['LOG_PATH'] = \
                                    f"{tbl_name}/{dq_name}"
                            else:
                                self.__dqcols.append(dqname)
                                resdataquality['COLNAME2'] = dq_name
                                resdataquality['LOG_PATH'] = \
                                    f"{tbl_name}/{dq_name}"

                            Writer.logger(f"{dqErrorCount} errors found for '{condition}' on '{dqname}'", True)

                            if self.__tableprops["DEBUG"]:
                                Writer.logger(self.__getDFSamples(dqTest))
                            Writer.qa_dataquality(dqTest, resdataquality['LOG_PATH'])
                            self.__ressummary['DATAQUALITY_TEST_FAILED'] += 1
                        else:
                            self.__ressummary['DATAQUALITY_TEST_PASSED'] += 1
                            Writer.logger(f"no errors found for {condition} on {dqname}", True)

                        resdataquality['END_STATUS'] = True
                        resdataquality['END_TIME'] = Atom_UDF.gettimestamp()

                    except Exception as e:
                        self.__ressummary['DATAQUALITY_TEST_FAILED'] += 1
                        resdataquality['END_STATUS'] = False
                        resdataquality['EXCEPTION'] = str(e)
                        et = Atom_UDF.gettimestamp()
                        resdataquality['END_TIME'] = et
                        self.__resdetailsummary['END_TIME'] = et
                        Writer.logger("exception in dataquality sql..", True)
                        Writer.logger(e)
                        Writer.logger("faulty sql")
                        Writer.logger(f"select {pkey}, '{columns}' as failed_col, '{tmp}' as condition, "
                                               f"{columns} as failed_colvalue from {tempView} where not ({condition})")

                    self.__resdetailsummary['DATAQUALITY'][str(self.__dqId)] = resdataquality
                    et = Atom_UDF.gettimestamp()
                    self.__resdetailsummary['END_TIME'] = et
                    gc.collect()
                    return resdataquality
                else:
                    Writer.logger("pkey/colname/condition/dqname cannot be empty.", True)
                    self.__resdetailsummary['DATAQUALITY'][str(self.__dqId)] = None
                    et = Atom_UDF.gettimestamp()
                    self.__resdetailsummary['END_TIME'] = et
                    gc.collect()
                    return None

            except Exception as e:
                print(str(e))
                del dqTest
                gc.collect()
                return None
        else:
            print(f"{TEST_NOT_INITIATED_MESSAGE} Project name < 3 chars & desc not set < 10 chars.")
            return None

    def finishtable(self):
        Writer.logger("completing table...", True)
        self.logTestRuns()
        #self.__report.add(self.__resdetailsummary) dont need this
        return self.__resdetailsummary

    #getreport file
    #def getreportfile(self):
    #    return self.__reportfile

    # close test
    def finishtest(self):
        Writer.logger("completing test...", True)
        self.__ressummary['END_TIME'] = Atom_UDF.gettimestamp()
        Writer.qa_testinfosummary(self.__ressummary)
        #self.__reportfile = self.__report.finishreport() dont need this
        return None #self.__reportfile dont need this

    # get log on exception
    def copylog(self):
        try:
            log_path = {self._settings.output_root}/{self._settings.applicationid}
            spark.read.format(self._settings.writeformat) \
                .load(f"{GFDOQA_TEST_PATH}/{self._settings.applicationid}/qalog").select("text").orderBy("id") \
                .coalesce(1).write.text(f"{log_path}/qalog")
            print(f"logs are copied to {log_path}/qalog.")
        except Exception as e:
            print(e)
        return


    # print sample evidence
    def printsampletestreport(self):
        Writer.logger("printing sample evidence...", True)
        fsr = self.getsummary()
        Writer.logger("\n")
        Writer.logger("\n")
        Writer.logger("Test Evidence (samples)")
        Writer.logger("=======================")
        #Writer.logger(f"Test report is saved in '{self.__reportfile}'") dot need this
        # print(fsr)
        Writer.logger("")
        #Writer.logger("Count Test")
        #Writer.logger("==========")
        #Writer.logger(f"Records in Source: {int(fsr['SRC_REC_COUNT'])}")
        #Writer.logger(f"Records in Target: {int(fsr['TRG_REC_COUNT'])}")
        #Writer.logger(f"Count Match "
        #       f"Test: {Atom_UDF.iif(int(fsr['SRC_REC_COUNT']) == int(fsr['TRG_REC_COUNT']), 'PASS', 'FAIL')}")
        #if int(fsr['SRC_REC_COUNT'])==0 and int(fsr['TRG_REC_COUNT'])==0:
        #    Writer.logger("since both source/target are 0 there isa chance data test is not done")
        #Writer.logger("")
        #Writer.logger("")

        # for cols
        spark = SparkSession.builder.getOrCreate()
        if self.__tableprops["COLTEST"]:
            if int(fsr['COLTEST_ERRORS']) > 0:
                try:
                    sevd = spark.read.format(self._settings.writeformat) \
                        .load(f"{self._settings.errorlog_base}/qa_metadatacomparison")
                    # sevd.createOrReplaceTempView("qa_metadatacomparison")
                    self._settings.createTempViews(sevd, "qa_metadatacomparison")

                    Writer.logger("Metadatacomparison Summary")
                    Writer.logger("==========================")
                    sevd = self._settings.sprk.sql(f"select tablename, count(1) as error_count from qa_metadatacomparison where "
                                    f"runid='{self._settings.applicationid}' group by tablename")
                    Writer.logger(f"Total errors logged :{sevd.count()}")
                    Writer.logger("first 20 are shown below...")
                    # sevd.show(20, truncate=False, vertical=True)
                    Writer.logger(self.__getDFSamples(sevd, 20))
                    Writer.logger("")
                    Writer.logger("Metadatacomparison Samples")
                    Writer.logger("==========================")
                    sevd = self._settings.sprk.sql(f"select * from qa_metadatacomparison where "
                                    f"runid='{self._settings.applicationid}'")
                    Writer.logger(f"Total errors logged :{sevd.count()}")
                    Writer.logger("first 20 are shown below...")
                    # sevd.show(20, truncate=False, vertical=True)
                    Writer.logger(self.__getDFSamples(sevd, 20))
                    Writer.logger("")
                except Exception as e:
                    Writer.logger(e)
            else:
                Writer.logger(f"No errors in column test.")
                Writer.logger("")
        else:
            Writer.logger(f"No column test done.")
            Writer.logger("")

        Writer.logger("")

        # for key
        if self.__tableprops["DATATEST"]:
            if int(fsr['KEYTEST_ERRORS']) > 0:
                print("\nKey test error part")
                try:
                    sevd = spark.read.format(self._settings.writeformat) \
                        .load(f"{self._settings.errorlog_base}/qa_keycomparison")
                    sevd.createOrReplaceTempView("qa_keycomparison")
                    #self._settings.createTempViews(sevd, "qa_keycomparison")
                    
                    Writer.logger("Keycomparison Summary")
                    Writer.logger("=====================")
                    sevd = self._settings.sprk.sql(
                        f"select tablename, error_msg, count(1) as error_count from qa_keycomparison "
                        f"where runid='{self._settings.applicationid}' "
                        f"group by tablename, error_msg"
                    )

                    Writer.logger(f"Total errors logged :{sevd.count()}")
                    Writer.logger("first 20 are shown below...")
                    # sevd.show(20, truncate=False, vertical=True)
                    Writer.logger(self.__getDFSamples(sevd, 20))

                    Writer.logger("Keycomparison Samples")
                    Writer.logger("=====================")
                    sevd = self._settings.sprk.sql(f"select * from qa_keycomparison where "
                                    f"runid='{self._settings.applicationid}'")

                    Writer.logger(f"Total errors logged :{sevd.count()}")
                    Writer.logger("first 20 are shown below...")
                    # sevd.show(20, truncate=False, vertical=True)
                    Writer.logger(self.__getDFSamples(sevd, 20))
                    Writer.logger("")
                except Exception as e:
                    print(e)
                    Writer.logger("")
            else:
                Writer.logger(f"No errors in key test.")
                Writer.logger("")
        else:
            Writer.logger(f"No key test done.")
            Writer.logger("")

        Writer.logger("")

        # for data
        if self.__tableprops["DATATEST"]:
            if int(fsr['DATATEST_ERRORS']) > 0:
                try:
                    sevd = spark.read.format(self._settings.writeformat) \
                        .load(f"{self._settings.errorlog_base}/qa_datacomparison")
                    # sevd.createOrReplaceTempView("qa_datacomparison")
                    self._settings.createTempViews(sevd, "qa_datacomparison")

                    Writer.logger("Datacomparison Summary")
                    Writer.logger("=====================")
                    sevd = self._settings.sprk.sql(f"select tablename, failedcol, count(1) as error_count from "
                                    f"qa_datacomparison where "
                                    f"runid='{self._settings.applicationid}' "
                                    f"group by tablename, failedcol")
                    Writer.logger(f"Total errors logged :{sevd.count()}")
                    Writer.logger("first 20 are shown below...")
                    # sevd.show(20, truncate=False, vertical=True)
                    Writer.logger(self.__getDFSamples(sevd, 20))
                    Writer.logger("Datacomparison Samples")
                    Writer.logger("=====================")
                    sevd = self._settings.sprk.sql(f"select * from qa_datacomparison where "
                                    f"runid='{self._settings.applicationid}'")
                    # sevd.show(20, truncate=False, vertical=True)
                    Writer.logger(self.__getDFSamples(sevd, 20))
                    Writer.logger("")
                except Exception as e:
                    Writer.logger(e)
            else:
                Writer.logger(f"No errors in data test.")
                Writer.logger("")
        else:
            Writer.logger(f"No data test done.")
            Writer.logger("")

        # dataquality test
        if self.__ressummary["DATAQUALITY_TESTS"]>0:
            if fsr['DATAQUALITY_ERRORS'] > 0:
                try:
                    Writer.logger("Dataquality Summary")
                    Writer.logger("=====================")
                    dqtests = self.__resdetailsummary["DATAQUALITY"]
                    for dqItem in dqtests:
                        dqtest = dqtests[dqItem]
                        if dqtest is not None:
                            Writer.logger(f"Columnname: '{dqtest['COLNAME']}'")
                            Writer.logger(f"Condition: '{dqtest['CONDITION']}'")
                            if dqtest['END_STATUS']:
                                if dqtest['DATAQUALITY_ERRORS'] > 0:
                                    Writer.logger(f"Total errors: {dqtest['DATAQUALITY_ERRORS']}, fail")
                                else:
                                    Writer.logger("No errors found, pass.")
                            else:
                                Writer.logger(f"Error: {dqtest['EXCEPTION']}")
                            Writer.logger("")
                except Exception as e:
                    Writer.logger(e)
            else:
                Writer.logger(f"No errors in dataquality test.")
                Writer.logger("")
        else:
            Writer.logger(f"No dataquality test done.")
            Writer.logger("")

        Writer.logger("end of sample evidence")
        Writer.logger("\n")
        Writer.logger("\n")


        # print sample code
        Writer.logger("#python code to get test errors & summary")
        # testruns, testinfo and testinfosummary
        Writer.logger("%python\n\n" \
               "#ATOM v{3}\n"
               "#optional\n"
               "#from pyspark.sql import SparkSession\n"\
               "#spark = SparkSession.builder.getOrCreate()\n\n"\
               "run_id='{2}'\n" \
               "#qa_testinfo\n" \
               "qa_testinfo_df=spark.read.format('{1}').load(f'{0}/{{run_id}}/summary_log/qa_testinfo')\n" \
               "qa_testinfo_df.createOrReplaceTempView('qa_testinfo')\n\n" \
               "#qa_testruns\n" \
               "qa_testruns_df=spark.read.format('{1}').load(f'{0}/{{run_id}}/summary_log/qa_testruns')\n" \
               "qa_testruns_df.createOrReplaceTempView('qa_testruns')\n\n" \
               "#qa_testinfosummary\n" \
               "qa_testinfosummary_df=spark.read.format('{1}').load(f'{0}/{{run_id}}/summary_log/qa_testinfosummary')\n" \
               "qa_testinfosummary_df.createOrReplaceTempView('qa_testinfosummary')\n"
               .format(self._settings.output_root, self._settings.writeformat, self._settings.applicationid, self._settings.CODE_VER))

        errlogsql = ""
        # metadata
        if self.__resdetailsummary['COL_TEST']['COLTEST_ERRORS']>0:
            Writer.logger("#qa_metadatacomparison\n" \
                   "qa_metadatacomparison_df=spark.read.format('{1}').load(f'{0}/{{run_id}}/error_log/qa_metadatacomparison')\n" \
                   "qa_metadatacomparison_df.createOrReplaceTempView(\"qa_metadatacomparison\")\n" \
                   .format(self._settings.output_root, self._settings.writeformat))
            errlogsql = f"select * from qa_metadatacomparison where runid='{self._settings.applicationid}';\n"

        # keytest
        if self.__resdetailsummary['KEY_TEST']['KEYTEST_ERRORS']>0:
            Writer.logger("#qa_keycomparison\n" \
                   "qa_keycomparison_df=spark.read.format('{1}').load(f'{0}/{{run_id}}/error_log/qa_keycomparison')\n" \
                   "qa_keycomparison_df.createOrReplaceTempView('qa_keycomparison')\n" \
                   .format(self._settings.output_root, self._settings.writeformat))
            errlogsql += f"select * from qa_keycomparison where runid='{self._settings.applicationid}';\n"

        # datatest
        if self.__resdetailsummary['DATA_TEST']['DATATEST_ERRORS']>0:
            Writer.logger("#qa_datacomparison\n" \
                   "qa_datacomparison_df=spark.read.format('{1}').load(f'{0}/{{run_id}}/error_log/qa_datacomparison')\n" \
                   "qa_datacomparison_df.createOrReplaceTempView('qa_datacomparison')\n" \
                   .format(self._settings.output_root, self._settings.writeformat))
            errlogsql += f"select * from qa_datacomparison where runid='{self._settings.applicationid}';\n"

        # dataquality test
        dqSQL = ""
        if self.__ressummary['DATAQUALITY_ERRORS'] > 0:
            Writer.logger("#qa_dataquality")
            dqtests = self.__resdetailsummary["DATAQUALITY"]
            for dqItem in dqtests:
                if dqtest is not None:
                    dqtest = dqtests[dqItem]
                    if dqtest['DATAQUALITY_ERRORS'] > 0:
                        Writer.logger(f"'#{dqtest['COLNAME']}'")
                        if dqtest['END_STATUS']:
                            Writer.logger("qa_dataquality_{2}=spark.read.format('{1}').load(f'{0}/{{run_id}}/error_log/qa_dataquality/{3}')\n" \
                             "qa_dataquality_{2}.createOrReplaceTempView('qa_dataquality_{2}')\n"
                            .format(self._settings.output_root, self._settings.writeformat,
                                    dqtest['COLNAME2'],dqtest['LOG_PATH']))

                            dqSQL += f"select * from qa_dataquality_{dqtest['COLNAME2']};\n"
                        else:
                            Writer.logger(dqtest['EXCEPTION'])

        Writer.logger("")
        Writer.logger("###################################################################################################")
        Writer.logger("%sql")
        Writer.logger("--summary")
        Writer.logger(f"select * from qa_testinfo where runid='{self._settings.applicationid}';")
        Writer.logger(f"select * from qa_testinfosummary where runid='{self._settings.applicationid}';")
        Writer.logger(f"select * from qa_testruns where runid='{self._settings.applicationid}';")
        Writer.logger("")
        if len(errlogsql.strip())>0:
            Writer.logger("--error logs")
            Writer.logger(errlogsql)
        else:
            Writer.logger("--no errors reported in key/data comparison  logs")
        Writer.logger("")

        if len(dqSQL.strip())>0:
            Writer.logger("--data quality logs")
            Writer.logger(dqSQL)
        else:
            Writer.logger("--no errors reported in data quality logs")

        log_path = f"{self._settings.output_root}/{self._settings.applicationid}"
        spark.read.format(self._settings.writeformat) \
            .load(f"{GFDOQA_TEST_PATH}/{self._settings.applicationid}/qalog").select("text").orderBy("id") \
            .coalesce(1).write.text(f"{log_path}/qalog")

        # self._settings.db_utils...
        #dbutils.fs.cp(self._settings.logfile, f"{log_path}")
        dbutils.fs.cp(f"{GFDOQA_TEST_PATH}/{self._settings.applicationid}", f"{log_path}", True)
        dbutils.fs.rm(f"{GFDOQA_TEST_PATH}/{self._settings.applicationid}", True)
        
        self._settings.logfile = ""

        Writer.logger("test completed...", True)
        return

    # log
    def logTestRuns(self):
        Writer.clear()
        # tablename
        Writer.add(self.__resdetailsummary['TABLE_NAME'])
        # starttime
        Writer.add(self.__resdetailsummary['START_TIME']["log"])
        # endtime
        Writer.add(self.__resdetailsummary['END_TIME']["log"])
        # metadata_test

        Writer.add(Atom_UDF.iif(self.__resdetailsummary['COL_TEST']['START_STATUS'] and
                           self.__resdetailsummary['COL_TEST']['END_STATUS'], 'Y', 'N'))
        # datacomparision_test
        Writer.add(Atom_UDF.iif((self.__resdetailsummary['KEY_TEST']['START_STATUS'] or
                            self.__resdetailsummary['DATA_TEST']['START_STATUS']) and
                           (self.__resdetailsummary['KEY_TEST']['END_STATUS'] or
                            self.__resdetailsummary['DATA_TEST']['END_STATUS']), 'Y', 'N'))
        # metadatatested_src
        Writer.add(Atom_UDF.iif(self.__resdetailsummary['COL_TEST']['START_STATUS'] and
                           self.__resdetailsummary['COL_TEST']['END_STATUS'],
                           str(self.__resdetailsummary['COL_TEST']['SRC_COL_COUNT']), '-1'))

        # metadatatested_trg
        Writer.add(Atom_UDF.iif(self.__resdetailsummary['COL_TEST']['START_STATUS'] and
                           self.__resdetailsummary['COL_TEST']['END_STATUS'],
                           str(self.__resdetailsummary['COL_TEST']['TRG_COL_COUNT']), '-1'))
        # metadata_errors
        Writer.add(Atom_UDF.iif(self.__resdetailsummary['COL_TEST']['START_STATUS'] and
                           self.__resdetailsummary['COL_TEST']['END_STATUS'],
                           str(self.__resdetailsummary['COL_TEST']['COLTEST_ERRORS']), '-1'))
        # colscount_src
        Writer.add(Atom_UDF.iif(self.__resdetailsummary['DATA_TEST']['START_STATUS'] and
                           self.__resdetailsummary['DATA_TEST']['END_STATUS'],
                           str(self.__resdetailsummary['DATA_TEST']['SRC_COL_COUNT']), '-1'))
        # recordcount_src
        Writer.add(Atom_UDF.iif(self.__resdetailsummary['DATA_TEST']['START_STATUS'] and
                           self.__resdetailsummary['DATA_TEST']['END_STATUS'],
                           str(self.__resdetailsummary['DATA_TEST']['SRC_REC_COUNT']), '-1'))
        # colscount_trg
        Writer.add(Atom_UDF.iif(self.__resdetailsummary['DATA_TEST']['START_STATUS'] and
                           self.__resdetailsummary['DATA_TEST']['END_STATUS'],
                           str(self.__resdetailsummary['DATA_TEST']['TRG_COL_COUNT']), '-1'))
        # recordcount_trg
        Writer.add(Atom_UDF.iif(self.__resdetailsummary['DATA_TEST']['START_STATUS'] and
                           self.__resdetailsummary['DATA_TEST']['END_STATUS'],
                           str(self.__resdetailsummary['DATA_TEST']['TRG_REC_COUNT']), '-1'))
        # key_errors
        Writer.add(Atom_UDF.iif(self.__resdetailsummary['KEY_TEST']['START_STATUS'] and
                           self.__resdetailsummary['KEY_TEST']['END_STATUS'],
                           str(self.__resdetailsummary['KEY_TEST']['KEYTEST_ERRORS']), '-1'))

        #failed cols
        Writer.add(Atom_UDF.iif(self.__resdetailsummary['DATA_TEST']['START_STATUS'] and
                           self.__resdetailsummary['DATA_TEST']['END_STATUS'],
                           str(self.__resdetailsummary['DATA_TEST']['FAILED_COLS']), '-1'))

        # data_errorrs
        Writer.add(Atom_UDF.iif(self.__resdetailsummary['DATA_TEST']['START_STATUS'] and
                           self.__resdetailsummary['DATA_TEST']['END_STATUS'],
                           str(self.__resdetailsummary['DATA_TEST']['DATATEST_ERRORS']), '-1'))

        # no of data quality test done
        Writer.add(Atom_UDF.iif(self.__ressummary['DATAQUALITY_TESTS'] == -1, '-1',
                           str(self.__ressummary['DATAQUALITY_TESTS'])))

        # no of data quality test passed
        Writer.add(Atom_UDF.iif(self.__ressummary['DATAQUALITY_TESTS'] == -1, '-1',
                           str(self.__ressummary['DATAQUALITY_TEST_PASSED'])))

        # no of data quality test failed
        Writer.add(Atom_UDF.iif(self.__ressummary['DATAQUALITY_TESTS'] == -1, '-1',
                           str(self.__ressummary['DATAQUALITY_TEST_FAILED'])))

        # data quality errors
        Writer.add(Atom_UDF.iif(self.__ressummary['DATAQUALITY_ERRORS'] == -1, '-1',
                           str(self.__ressummary['DATAQUALITY_ERRORS'])))
        # runid
        Writer.add(self._settings.applicationid)
        Writer.qa_testruns()
        return


"""
#issues
when date is given as pk need to rep the issue
nimpha issues need to rep the issue
variable in python script---- done--- tobe tested
"""

"""
Geethika: column count mismatch showing as 2 but there are more cols
Jagrati: no pk first col is null and it does not concat but concat_ws works
Veera: count mis matches 
Muneer: sharepoint. TBD
Shilpi: ade done
"""


#testrunner.py

#writer.py
class Writer:
    # writeFormat = "delta"
    _settings:Optional[SettingsProto] = None
    linenum = 0
    fields = []
    logger_schema = StructType([StructField("id", StringType()),
                         StructField("text", StringType())])

    qa_testinfo_schema = StructType([StructField("runid", StringType()),
                                     StructField("starttime", StringType()),
                                     StructField("project", StringType()),
                                     StructField("project_desc", StringType()),
                                     StructField("username", StringType()),
                                     StructField("ver", StringType()),
                                     StructField("runyear", StringType())])

    qa_testruns_schema = StructType([StructField("tablename", StringType()),
                                     StructField("starttime", StringType()),
                                     StructField("endtime", StringType()),
                                     StructField("metadata_test", StringType()),
                                     StructField("datacomparision_test", StringType()),
                                     StructField("metadatatested_src", StringType()),
                                     StructField("metadatatested_trg", StringType()),
                                     StructField("metadata_errors", StringType()),
                                     StructField("colscount_src", StringType()),
                                     StructField("recordcount_src", StringType()),
                                     StructField("colscount_trg", StringType()),
                                     StructField("recordcount_trg", StringType()),
                                     StructField("key_errors", StringType()),
                                     StructField("failedcols_datatest", StringType()),
                                     StructField("data_errors", StringType()),
                                     StructField("dataquality_tests", StringType()),
                                     StructField("dataquality_test_passed", StringType()),
                                     StructField("dataquality_test_failed", StringType()),
                                     StructField("dataquality_errors", StringType()),
                                     StructField("runid", StringType())])

    qa_testinfosummary_schema = StructType([StructField("runid", StringType()),
                                            StructField("starttime", StringType()),
                                            StructField("endtime", StringType())])
    """
    qa_testinfosummary_schema = StructType([StructField("runid", StringType()),
                                            StructField("testsuitename", StringType()),
                                            StructField("starttime", StringType()),
                                            StructField("endtime", StringType()),
                                            StructField("tablestested", StringType()),
                                            StructField("metadata_src_cols", StringType()),
                                            StructField("metadata_trg_cols", StringType()),
                                            StructField("metadata_errors", StringType()),
                                            StructField("colscount_src", StringType()),
                                            StructField("recordcount_src", StringType()),
                                            StructField("colscount_trg", StringType()),
                                            StructField("recordcount_trg", StringType()),
                                            StructField("key_errors", StringType()),
                                            StructField("data_errors", StringType()),
                                            StructField("dataquality_errors", StringType()),
                                            StructField("runyear", StringType())])
    """

    @classmethod
    def set_settings(cls, settings:SettingsProto) -> None:
        cls._settings = settings
        return

    @staticmethod
    def add(o):
        Writer.fields.append(o)
        return

    @staticmethod
    def clear():
        Writer.fields = []

    @classmethod
    def logger(cls, log, printtoconsole=False):
        try:
            logLevel = cls._settings.loglevel
            if printtoconsole:
                print(log)

            if logLevel.lower() in ['all', 'summary']:
                path = f"{GFDOQA_TEST_PATH}/{cls._settings.applicationid}/qalog"
                records = [[str(Writer.linenum), log]]
                cls._settings.sprk.createDataFrame(records, Writer.logger_schema) \
                    .withColumn("id", col("id").cast(LongType())) \
                    .write \
                    .format(cls._settings.writeformat) \
                    .mode("append") \
                    .save(path)
                
                Writer.linenum += 1

        except Exception as e:
            print(e)
        return

    @classmethod
    def qa_testinfo(cls):
        try:
            logLevel = cls._settings.loglevel
            if logLevel.lower() in ['all', 'summary']:
                print("writing to qa_testinfo...")
                records = [Writer.fields]
                Writer.fields = []

                cls._settings.sprk.createDataFrame(records, Writer.qa_testinfo_schema) \
                    .withColumn("starttime", col("starttime").cast(TimestampType())) \
                    .withColumn("runyear", col("runyear").cast(LongType())) \
                    .write.partitionBy("runyear") \
                    .format(cls._settings.writeformat) \
                    .mode("append") \
                    .save(f"{cls._settings.summarylog_base}/qa_testinfo")
            else:
                print("writing to qa_testinfo skipped...")
        except Exception as e:
            Writer.logger("exception in writer.qa_testinfo")
            Writer.logger(e)
        return

    @classmethod
    def qa_testruns(cls):
        try:
            logLevel = cls._settings.loglevel
            if logLevel.lower() in ['all', 'summary']:
                print("writing to qa_testruns...")
                records = [Writer.fields]
                Writer.fields = []

                cls._settings.sprk.createDataFrame(records, Writer.qa_testruns_schema) \
                    .withColumn("starttime", col("starttime").cast(TimestampType())) \
                    .withColumn("endtime", col("endtime").cast(TimestampType())) \
                    .withColumn("metadatatested_src", col("metadatatested_src").cast(LongType())) \
                    .withColumn("metadatatested_trg", col("metadatatested_trg").cast(LongType())) \
                    .withColumn("metadata_errors", col("metadata_errors").cast(LongType())) \
                    .withColumn("colscount_src", col("colscount_src").cast(LongType())) \
                    .withColumn("recordcount_src", col("recordcount_src").cast(LongType())) \
                    .withColumn("colscount_trg", col("colscount_trg").cast(LongType())) \
                    .withColumn("recordcount_trg", col("recordcount_trg").cast(LongType())) \
                    .withColumn("key_errors", col("key_errors").cast(LongType())) \
                    .withColumn("failedcols_datatest", col("failedcols_datatest").cast(LongType())) \
                    .withColumn("data_errors", col("data_errors").cast(LongType())) \
                    .withColumn("dataquality_tests", col("dataquality_tests").cast(LongType())) \
                    .withColumn("dataquality_test_passed", col("dataquality_test_passed").cast(LongType())) \
                    .withColumn("dataquality_test_failed", col("dataquality_test_failed").cast(LongType())) \
                    .withColumn("dataquality_errors", col("dataquality_errors").cast(LongType())) \
                    .write.partitionBy("runid") \
                    .format(cls._settings.writeformat) \
                    .mode("append") \
                    .save(f"{cls._settings.summarylog_base}/qa_testruns")
            else:
                print("writing to qa_testruns skipped...")
        except Exception as e:
            Writer.logger("exception in writer.qa_testruns")
            Writer.logger(e)
        return

    @classmethod
    def qa_testinfosummary(cls, summarInfo):
        try:
            logLevel = cls._settings.loglevel
            if logLevel.lower() in ['all', 'summary']:
                print("writing to qa_testinfosummary...")
                Writer.clear()
                Writer.add(cls._settings.applicationid)
                #Writer.add(summarInfo['FILE_NAME'])
                Writer.add(summarInfo['START_TIME']["log"])
                Writer.add(summarInfo['END_TIME']["log"])
                #Writer.add(summarInfo['TABLE_COUNT'])
                #Writer.add(summarInfo['MD_SRC_COL_COUNT'])
                #Writer.add(summarInfo['MD_TRG_COL_COUNT'])
                #Writer.add(summarInfo['COLTEST_ERRORS'])
                #Writer.add(summarInfo['DT_SRC_COL_COUNT'])
                #Writer.add(summarInfo['SRC_REC_COUNT'])
                #Writer.add(summarInfo['DT_TRG_COL_COUNT'])
                #Writer.add(summarInfo['TRG_REC_COUNT'])
                #Writer.add(summarInfo['KEYTEST_ERRORS'])
                #Writer.add(summarInfo['DATATEST_ERRORS'])
                #Writer.add(summarInfo['DATAQUALITY_ERRORS'])
                #Writer.add(summarInfo['START_TIME']["log"][:4])

                records = [Writer.fields]
                Writer.fields = []
                cls._settings.sprk.createDataFrame(records, Writer.qa_testinfosummary_schema) \
                    .withColumn("starttime", col("starttime").cast(TimestampType())) \
                    .withColumn("endtime", col("endtime").cast(TimestampType())) \
                    .write.partitionBy("runid") \
                    .format(cls._settings.writeformat) \
                    .mode("append") \
                    .save(f"{cls._settings.summarylog_base}/qa_testinfosummary")

                """
                cls._settings.sprk.createDataFrame(cls._settings.sprk.sparkContext.parallelize(records), Writer.qa_testinfosummary_schema) \
                    .withColumn("starttime", col("starttime").cast(TimestampType())) \
                    .withColumn("endtime", col("endtime").cast(TimestampType())) \
                    .withColumn("tablestested", col("tablestested").cast(LongType())) \
                    .withColumn("metadata_src_cols", col("metadata_src_cols").cast(LongType())) \
                    .withColumn("metadata_trg_cols", col("metadata_trg_cols").cast(LongType())) \
                    .withColumn("metadata_errors", col("metadata_errors").cast(LongType())) \
                    .withColumn("colscount_src", col("colscount_src").cast(LongType())) \
                    .withColumn("recordcount_src", col("recordcount_src").cast(LongType())) \
                    .withColumn("colscount_trg", col("colscount_trg").cast(LongType())) \
                    .withColumn("recordcount_trg", col("recordcount_trg").cast(LongType())) \
                    .withColumn("key_errors", col("key_errors").cast(LongType())) \
                    .withColumn("data_errors", col("data_errors").cast(LongType())) \
                    .withColumn("dataquality_errors", col("dataquality_errors").cast(LongType())) \
                    .withColumn("runyear", col("runyear").cast(LongType())) \
                    .write.partitionBy("runid") \
                    .format(cls._settings.writeformat) \
                    .mode("append") \
                    .save(f"{cls._settings.summarylog_base}/qa_testinfosummary")
                """
            else:
                print("writing to qa_testinfosummary skipped...")
        except Exception as e:
            Writer.logger("exception in writer.qa_testinfosummary")
            Writer.logger(e)

    @classmethod
    def qa_metadatacomparison(cls, dataTable, tp):
        try:
            logLevel = cls._settings.loglevel
            if logLevel.lower() == 'all':
                print("writing to qa_metadatacomparison...")
                tableName = tp["TABLENAME"]  # Table.name
                appID = cls._settings.applicationid
                maxError = cls._settings.maxerror
                cls._settings.createTempViews(dataTable, f"col_errors_{tableName}")

                dataTable.withColumn("tablename", F.lit(tableName)).withColumn("runid", F.lit(appID)) \
                    .write.format(cls._settings.writeformat) \
                    .partitionBy('runid') \
                    .mode('append') \
                    .save(f"{cls._settings.errorlog_base}/qa_metadatacomparison")
            else:
                print("writing to qa_metadatacomparison skipped...")

            cls._settings.cleartempview(f"col_errors_{tableName}")
            del dataTable
            gc.collect()
        except Exception as e:
            del dataTable
            gc.collect()
            Writer.logger("exception in writer.qa_metadatacomparison")
            Writer.logger(e)

    @classmethod
    def qa_keycomparison(cls, dataTable, tp):
        try:
            logLevel = cls._settings.loglevel
            if logLevel.lower() == 'all':
                print("writing to qa_keycomparison...")
                tableName = tp["TABLENAME"]  # Table.name
                appID = cls._settings.applicationid
                maxError = cls._settings.maxerror

                cls._settings.setsparkobject(SparkSession.builder.getOrCreate())
                dataTable.createOrReplaceTempView(f"key_errors_{tableName}")
                #cls._settings.createTempViews(dataTable, f"key_errors_{tableName}")
                if int(maxError) < 0:
                    print("Creating tmp df for results in qa_keycomparison")
                    dataTable.withColumnRenamed("keycol", "pkey").withColumnRenamed("errortype", "error_msg") \
                        .withColumn("runid", F.lit(appID)).withColumn("tablename", F.lit(tableName)) \
                        .write.partitionBy('runid').format(cls._settings.writeformat) \
                        .mode('append').save(f"{cls._settings.errorlog_base}/qa_keycomparison")
                else:
                    spark.sql("with k as (select row_number() over(partition by  a.errortype order by "
                             f"a.errortype) as rn, '{tableName}' as tablename, "
                             "cleantabledelimiters(a.keycol) as pkey, "
                             f"cleantabledelimiters(a.errortype) as error_msg, '{appID}' as runid "
                             f"from key_errors_{tableName} a) select tablename, pkey, error_msg, runid from k "
                             f"where rn<={str(maxError).replace(',', '')}").write.partitionBy('runid') \
                        .format(cls._settings.writeformat).mode('append') \
                        .save(f"{cls._settings.errorlog_base}/qa_keycomparison")
            else:
                print("writing to qa_keycomparison skipped...")

            cls._settings.cleartempview(f"key_errors_{tableName}")
            del dataTable
            gc.collect()
        except Exception as e:
            del dataTable
            gc.collect()
            Writer.logger("exception in writer.qa_keycomparison")
            Writer.logger(e)
            print(e)
        return

    @classmethod
    def qa_datacomparison(cls, dataTable, tp):
        try:
            logLevel = cls._settings.loglevel
            if logLevel.lower() == 'all':
                print("writing to qa_datacomparison...")
                tableName = tp["TABLENAME"]  # Table.name
                appID = cls._settings.applicationid
                maxError = cls._settings.maxerror

                # dataTable.createOrReplaceTempView(f"data_errors_{tableName}")
                cls._settings.createTempViews(dataTable, f"data_errors_{tableName}")

                if int(maxError) < 0:
                    dataTable.withColumnRenamed("field_name", "failedcol") \
                        .withColumnRenamed("keycol", "pkey") \
                        .withColumnRenamed("source_data", "src_data") \
                        .withColumnRenamed("target_data", "trg_data") \
                        .withColumn("runid", F.lit(appID)) \
                        .withColumn("tablename", F.lit(tableName)) \
                        .write.format(cls._settings.writeformat) \
                        .partitionBy('runid') \
                        .mode('append') \
                        .save(f"{cls._settings.errorlog_base}/qa_datacomparison")
                else:
                    cls._settings.sprk.sql(f"with d as (select row_number() over(partition by  "
                             f"a.field_name order by a.field_name) as rn, '{tableName}' as tablename, "
                             f"cleantabledelimiters(a.field_name) as failedcol, "
                             f"cleantabledelimiters(a.keycol) as pkey, "
                             f"cleantabledelimiters(a.source_data) as src_data, "
                             f"cleantabledelimiters(a.target_data) as trg_data, '{appID}' as runid "
                             f"from data_errors_{tableName} a) "
                             f"select tablename, failedcol, pkey, src_data, trg_data, runid from d "
                             f"where rn<={str(maxError).replace(',', '')}") \
                        .write.format(cls._settings.writeformat) \
                        .partitionBy('runid') \
                        .mode('append') \
                        .save(f"{cls._settings.errorlog_base}/qa_datacomparison")
            else:
                print("writing to qa_datacomparison skipped...")

            cls._settings.cleartempview(f"data_errors_{tableName}")
            del dataTable
            gc.collect()
        except Exception as e:
            del dataTable
            gc.collect()
            Writer.logger("exception in writer.qa_datacomparison")
            Writer.logger(e)
        return

    @classmethod
    def qa_dataquality(cls, dataTable, dqname):
        try:
            logLevel = cls._settings.loglevel
            if logLevel.lower() == 'all':
                print("writing to qa_dataquality...")
                maxError = cls._settings.maxerror

                if int(maxError) < 0:
                    dataTable.write \
                        .format(cls._settings.writeformat) \
                        .mode('append') \
                        .save(f"{cls._settings.errorlog_base}/qa_dataquality/{dqname}")
                else:
                    dataTable.limit(int(maxError)).write \
                        .format(cls._settings.writeformat) \
                        .mode('append') \
                        .save(f"{cls._settings.errorlog_base}/qa_dataquality/{dqname}")
            else:
                print("writing to qa_dataquality skipped...")

            del dataTable
            gc.collect()
        except Exception as e:
            del dataTable
            gc.collect()
            Writer.logger("exception in writer.qa_dataquality")
            Writer.logger(e)


#writer.py